In [1]:
from pathlib import Path
import os
print(os.getcwd())
print('\n'.join(str(x) for x in Path('/workspace').iterdir()))

/workspace
/workspace/lost+found
/workspace/.kernel_llm_logs_1.jsonl
/workspace/.prompts
/workspace/.skills
/workspace/finance_rules.md
/workspace/memory
/workspace/.config
/workspace/finance_audit_sample.csv


In [2]:
print(Path('finance_rules.md').read_text()); print(Path('finance_audit_sample.csv').read_text())

# Financial details of companies built on popular open-source projects

Part of a pre-registered study (issue #380). The companies in your input were already classified as `core`: their
main paid product is an open-source project they lead, in another form (hosted service, open-core or enterprise
edition, paid licences or support, or hardware). Your job is to find what each company raised, what it was or is
worth, and what happened to it, **with a source URL for every number**.

## Input

A CSV with one row per company: `owner` (GitHub login), `entity` (company name from the classification),
`parent` (current owner if acquired, when known), `country`, `basis`, `status_hint`, `stars` (the org's stars in
the frame) and the top repositories. Rows with `h9_addition=1` are companies that built the project and later
handed it to a foundation (e.g. Anyscale and Ray); research the company named in `entity`, not the foundation.
Several GitHub orgs can belong to one company (e.g. Docker Inc. own

In [3]:
import pandas as pd, json, datetime, asyncio
src=pd.read_csv('finance_audit_sample.csv').fillna('')
print(src[['github_orgs','entity','website']].to_string(index=True));print('UTC',datetime.datetime.now(datetime.timezone.utc).isoformat())
Path('audit_work').mkdir(exist_ok=True)

     github_orgs                           entity                         website
0      anomalyco                          Anomaly                https://anoma.ly
1       Kilo-Org                        Kilo Code                                
2     openbq-org                      OpenBB Inc.                                
3    datalab-org           datalab industries ltd         https://datalab-org.io/
4      ankitects   Anki Software, LLC (Ankitects)                                
5        notable       Notable (Fabio Spampinato)             https://notable.app
6    sinaptik-ai              Sinaptik (PandasAI)           https://pandas-ai.com
7      VectifyAI                       Vectify AI            https://pageindex.ai
8       usebruno               Bruno Software Inc       https://www.usebruno.com/
9       marmelab                         marmelab             http://marmelab.com
10  BruceDevices                    Bruce Devices                  bruce.computer
11      tinygrad

In [4]:
from e14c.subagents import submit_task
common='''Independently research public financial disclosures for assigned rows in /workspace/finance_audit_sample.csv, following /workspace/finance_rules.md. Output exactly these columns: github_orgs,entity,status,status_date,acquirer,exit_value_usd,exit_source,total_funding_usd,funding_asof,funding_source,last_valuation_usd,valuation_date,valuation_source,peak_value_usd,peak_value_date,peak_source,ever_1b,notes. Notes <=25 words. No ticker/legal name. Use 3-5 web searches/fetches per company approximately, authoritative company/investor/news sources, not aggregator guesses. Numbers blank unless source verifies. Sum explicitly disclosed equity rounds only if no cumulative figure, exclude debt, distinguish funding from valuation. Private status_date blank as not an event. A number's source URL must state it (if sum use pipe-separated component URLs and document arithmetic in evidence). Identify conflicts and uncertainty. Save full BibTeX for every used web source using e14c.web.save_citation. Keep source excerpts and dates in an evidence JSON or markdown alongside CSV. All files in assigned folder, do not touch root manifest. Return file paths, key findings/conflicts. Current environment UTC 2026-11-26; do not invent later updates beyond located sources. Check current status. Read available input first. '''
groups={'smallA':[3,4,5,6,7,8,9,10,17,18], 'smallB':[20,21,22,23,24,28,30,34,35,36,37]}
tasks={}
for key, inds in groups.items():
    folder=f'/workspace/audit_work/{key}'
    Path(folder).mkdir(exist_ok=True)
    tasks[key]=await submit_task(common+f' Assigned ZERO-BASED indices {inds}. Save CSV at {folder}/rows.csv and evidence at {folder}/evidence.md (or JSON). You have about 30 minutes; prioritize complete, conservative evidence-backed rows.',agent_type='data-retrieval',project_id=os.environ.get('PROJECT_ID'),timeout=2200)
print(tasks)

{'smallA': '7c97750e-fecb-4302-badb-3b7859ce2103', 'smallB': '2a3ea50c-35eb-4da3-8ff6-70193cc99b97'}


In [5]:
from e14c.web import web_search, fetch_parsed_page, save_citation
searches={}; pages={}; used={}
async def search(label,query,n=5):
    r=await web_search(query,num_results=n)
    searches[label]={'query':query,'results':r}
    Path('audit_work/searches.json').write_text(json.dumps(searches,indent=2,default=str))
    print('\n###',label,query)
    for i,x in enumerate(r):
        print(i,x['title'],x['url'],str(x.get('publishedDate','')),'\n',x.get('full_text','')[:2500] or x.get('snippet',''))
    return r
r=await asyncio.gather(search('anomaly','Anomaly opencode SST funding valuation 2026 2025'),search('kilo','Kilo Code funding raised valuation 2026'),search('openbb','OpenBB funding raised total 2026 acquisition'))


### anomaly Anomaly opencode SST funding valuation 2026 2025
0 Anomaly | Commercial Open Source Software | Cossmology https://cossmology.com/organizations/sst None 
 Anomaly | Commercial Open Source Software | Cossmology

# Anomaly

Open source dev tools. Maintainers of OpenCode.

San Francisco, CA, USA Founded 2017

Anomaly, short for Anomaly Innovations Inc., is a software company that builds open-source developer tools. Anomaly is the provider of the open source AI coding agent OpenCode.

### Websites:

https://anoma.ly/ https://sst.dev/ https://opentui.com/ https://opencode.ai/ https://models.dev/

Last Updated: August 7, 2026

## Current Valuation

No valuation data available

## Funding Summary

$1.1M

Total reported funding

Microsoft Newsroom: COSS Companies Sign Open Letter Urging US to Encourage Development of Open Weight Models

Media Mention June 12, 2026

Developers Digest: OpenCode Developer Guide: The Open Source AI Coding Agent with 160K Stars - Developers Digest

Anno


### kilo Kilo Code funding raised valuation 2026
0 We raised $8 Million to bring Kilo Speed to Agentic Engineering https://blog.kilo.ai/p/kilo-raised-8-million-seed-round 2025-12-10T14:05:45.000Z 
 We raised $8 Million to bring Kilo Speed to Agentic Engineering

# We raised $8 Million to bring Kilo Speed to Agentic Engineering

Scott Breitenother

Dec 10, 2025

Today we announced that Kilo Code has raised $8m in seed funding. The funding round was led by Cota Capital with participation from Breakers, General Catalyst, Quiet Capital, and Tokyo Black. This is an exciting moment for us and I want to pause and share why I am personally so energized about Kilo.

My last venture was building Brooklyn Data, a consultancy that helped companies unlock the value of their data. I started Brooklyn Data in 2018 and we rode a wave that was far bigger than we were: companies adopting cloud data warehouses like Snowflake, the modern data stack including dbt, and the rise of analytics engineering. It 

In [6]:
def show(label,limit=4200):
    print('\n###',label)
    for i,x in enumerate(searches[label]['results']):
        print(i,x['title'],x['url'],x.get('publishedDate'), '\n',(x.get('full_text') or x.get('snippet') or '')[:limit])
show('kilo');show('openbb',3200);show('anomaly',800)


### kilo
0 We raised $8 Million to bring Kilo Speed to Agentic Engineering https://blog.kilo.ai/p/kilo-raised-8-million-seed-round 2025-12-10T14:05:45.000Z 
 We raised $8 Million to bring Kilo Speed to Agentic Engineering

# We raised $8 Million to bring Kilo Speed to Agentic Engineering

Scott Breitenother

Dec 10, 2025

Today we announced that Kilo Code has raised $8m in seed funding. The funding round was led by Cota Capital with participation from Breakers, General Catalyst, Quiet Capital, and Tokyo Black. This is an exciting moment for us and I want to pause and share why I am personally so energized about Kilo.

My last venture was building Brooklyn Data, a consultancy that helped companies unlock the value of their data. I started Brooklyn Data in 2018 and we rode a wave that was far bigger than we were: companies adopting cloud data warehouses like Snowflake, the modern data stack including dbt, and the rise of analytics engineering. It was an incredible moment. The foundation

In [7]:
for label in ['kilo','openbb']:
 print('\n',label)
 for i,x in enumerate(searches[label]['results']):
  print(i,x['url'],(x.get('full_text') or '')[:1300])



 kilo
0 https://blog.kilo.ai/p/kilo-raised-8-million-seed-round We raised $8 Million to bring Kilo Speed to Agentic Engineering

# We raised $8 Million to bring Kilo Speed to Agentic Engineering

Scott Breitenother

Dec 10, 2025

Today we announced that Kilo Code has raised $8m in seed funding. The funding round was led by Cota Capital with participation from Breakers, General Catalyst, Quiet Capital, and Tokyo Black. This is an exciting moment for us and I want to pause and share why I am personally so energized about Kilo.

My last venture was building Brooklyn Data, a consultancy that helped companies unlock the value of their data. I started Brooklyn Data in 2018 and we rode a wave that was far bigger than we were: companies adopting cloud data warehouses like Snowflake, the modern data stack including dbt, and the rise of analytics engineering. It was an incredible moment. The foundational technology changed, the culture of data changed, and those two changes created a cascading 

In [8]:
r=await asyncio.gather(search('anomaly2','SST raises million seed funding 2021 serverless stack'),search('kilo2','Anaconda acquires Kilo July 2026'),search('openbb2','OpenBB 8.5 million seed 2022 valuation'),search('openbb3','Didier Lopes OpenBB winding down August 2026'))


### openbb2 OpenBB 8.5 million seed 2022 valuation
0 OpenBB Raises $8.5M in Seed Round Funding Following Open https://www.globenewswire.com/news-release/2022/03/29/2412195/0/en/openbb-raises-8-5m-in-seed-round-funding-following-open-source-project-gamestonk-terminal-s-success.html 2022-03-29T16:00:00.000Z 
 OpenBB Raises $8.5M in Seed Round Funding Following Open

# OpenBB Raises $8.5M in Seed Round Funding Following Open Source Project Gamestonk Terminal's Success

## Gamestonk Terminal Renamed OpenBB Terminal to Reflect Big Aspirations After $8.5M Seed Investment Round

 March 29, 2022 12:00 ET | Source: OpenBB 

DELAWARE CITY, Del., March 29, 2022 (GLOBE NEWSWIRE) -- Following the success of their open source project, Gamestonk Terminal, the founding team composed of Didier Lopes, James Maslek and Artem Veremey is launching a new company: OpenBB. 

OpenBB is partnering with OSS Capital to build world-class investment research products. OSS Capital is leading the seed round alongsid


### anomaly2 SST raises million seed funding 2021 serverless stack
0 SST raises $1M | SST https://sst.dev/blog/serverless-stack-raises-1m-to-make-it-easy-to-build-serverless-apps/ 2021-07-23T00:00:00.000Z 
 SST raises $1M | SST

# SST raises $1M

Jay July 23, 2021

Today we are announcing our $1M seed round. The new funding helps us in our mission to make it easy for developers to build full-stack serverless applications. You can read more about this announcement over on TechCrunch as well.

Our SST allows developers to test and make changes to their applications by directly connecting their local machines to the cloud. The biggest issue that developers face while adopting serverless is the ability to debug their applications locally. With SST, developers are able to set breakpoints and inspect their code, just as they would with any other type of application. Since its public launch 5 months ago, SST has grown to over 2K stars on GitHub and has been downloaded over 60K times.

We wou

In [9]:
for label in ['anomaly2','kilo2','openbb3']:
 print('\n###',label)
 for i,x in enumerate(searches[label]['results']):
  print(i,x['title'],x['url'], '\n',(x.get('full_text') or '')[:1600])


### anomaly2
0 SST raises $1M | SST https://sst.dev/blog/serverless-stack-raises-1m-to-make-it-easy-to-build-serverless-apps/ 
 SST raises $1M | SST

# SST raises $1M

Jay July 23, 2021

Today we are announcing our $1M seed round. The new funding helps us in our mission to make it easy for developers to build full-stack serverless applications. You can read more about this announcement over on TechCrunch as well.

Our SST allows developers to test and make changes to their applications by directly connecting their local machines to the cloud. The biggest issue that developers face while adopting serverless is the ability to debug their applications locally. With SST, developers are able to set breakpoints and inspect their code, just as they would with any other type of application. Since its public launch 5 months ago, SST has grown to over 2K stars on GitHub and has been downloaded over 60K times.

We would like to take this opportunity to thank SST community for being involved righ

In [10]:
show('kilo2',2600)
print('\nOPENBB primary urls');print([(i,x['url']) for i,x in enumerate(searches['openbb3']['results'])])


### kilo2
0 Anaconda Acquires Kilo Code to Power the Trillion-Token Enterprise | Anaconda https://www.anaconda.com/press/anaconda-acquires-kilo-code 2026-07-15T14:02:05.000Z 
 Anaconda Acquires Kilo Code to Power the Trillion-Token Enterprise | Anaconda

# Anaconda Acquires Kilo Code to Power the Trillion-Token Enterprise

- July 15, 2026 

Acquisition brings Kilo’s model-agnostic, agentic engineering platform already trusted by 3 million developers, extending Anaconda’s foundation for AI-native development

AUSTIN, Texas – July 15, 2026 – Every enterprise wants to get a return on investment on their AI investments, however 80% of AI projects fail to make it to production. The problem isn’t whether or not to leverage AI, the hard problem is doing that safely, and affordably, at scale. Anaconda Inc., the trusted foundation for AI-native development, today announced it has acquired Kilo Code (aka “Kilo”), the open source, model-agnostic agentic engineering platform used by more than 3 m

In [11]:
def cite_result(label,i,why=''):
 x=searches[label]['results'][i];url=x['url']
 if url not in used:
  key='audit'+str(len(used)+1)
  title=x['title'].replace('&',r'\&').replace('%',r'\%').replace('_',r'\_').replace('#',r'\#')
  from urllib.parse import urlparse
  author=urlparse(url).netloc
  bib=f'@online{{{key},\n author = {{{{{author}}}}},\n title = {{{title}}},\n url = {{{url}}},\n urldate = {{2026-11-26}},\n'
  date=x.get('publishedDate')
  if date:bib+=f' date = {{{date[:10]}}},\n'
  bib+='}\n';save_citation(bib)
  used[url]={**x,'use':why,'key':key}
  Path('audit_work/used_sources.json').write_text(json.dumps(used,indent=2,default=str))
 return url
async def fetch(label,url):
 text=await fetch_parsed_page(url);pages[label]={'url':url,'text':text}
 Path('audit_work/pages.json').write_text(json.dumps(pages,indent=2))
 print(label,text[:14000]);return text
cols='github_orgs entity status status_date acquirer exit_value_usd exit_source total_funding_usd funding_asof funding_source last_valuation_usd valuation_date valuation_source peak_value_usd peak_value_date peak_source ever_1b notes'.split()
rows={}
def put(idx,**kw):
 d={c:'' for c in cols};d.update(src.iloc[idx][['github_orgs','entity']].to_dict());d.update(kw);rows[idx]=d
 pd.DataFrame([rows[i] for i in sorted(rows)],columns=cols).to_csv('audit_work/my_rows.csv',index=False)
put(0,status='private',total_funding_usd='1000000',funding_asof='2021-07',funding_source=cite_result('anomaly2',0),ever_1b='no',notes='Only announced seed counted; aggregator $1.1m total not independently verified.')
put(1,status='acquired',status_date='2026-07',acquirer='Anaconda',exit_source=cite_result('kilo2',0),total_funding_usd='8000000',funding_asof='2025-12',funding_source=cite_result('kilo',0),ever_1b='no',notes='Acquisition terms undisclosed; no disclosed valuation found.')
put(2,status='shut_down',status_date='2026-08',total_funding_usd='8500000',funding_asof='2022-03',funding_source=cite_result('openbb2',0),ever_1b='no',notes='Company winding down; product suite open-sourced, not an acquisition.')
cite_result('openbb3',0,'Shutdown announcement')
r=await asyncio.gather(search('tiny','Tiny Corp tinygrad funding raised 2026 million'),search('penpot','Kaleidos Penpot total funding raised 2026'),search('crew','CrewAI funding total valuation 2026'),search('continue','Continue Dev funding total valuation 2026'),search('unsloth','Unsloth AI funding valuation 2026'))


### crew CrewAI funding total valuation 2026
0 CrewAI Company Profile: Funding, Financials and Investors | Airframe https://www.airframe.ai/product/crewai-com/co-diligence None 
 CrewAI Company Profile: Funding, Financials and Investors | Airframe

# CrewAI

AI Native

Build a crew of AI agents today, scale everything tomorrow.

Est 2024| Valuation $185M| Raised $44.5M| 51 Ref Customers

LinkedIn 103K| Web traffic 343K| FTEs 30| HQ San Francisco, US

Markets Agent Frameworks Agent Builder Platforms

Published Nov 10, 2026

## 01 Organization Info

Founded 2024

Headcount 30

## 02 Financial Stability

Total Raised $44.5M

Last Round Sep 2024

Valuation $185M

## 05 Investors

Andrew Ng · Blitzscaling Ventures · Boldstart Ventures · Craft Ventures · Dallas Venture Capital · Dharmesh Shah · Earl Grey Capital · FirsthandVC · Flybridge (New York) · Grace (Other Commercial Services) · Insight Partners (New York) · Next Wave NYC · Prototype Capital (San Francisco) · Punch Capital · SineWave


### unsloth Unsloth AI funding valuation 2026
0 Unsloth AI company information, funding & investors | Dealroom.co https://app.dealroom.co/companies/unsloth_ai None 
 Unsloth AI company information, funding & investors | Dealroom.co

# Unsloth AI

Open-source software for local AI training and inference.

HQ location

San Francisco, United States

Website unsloth.ai

Launch date

2023

Employees 11-50 people

Enterprise value

$500—750k

https://x.com/unslothai https://www.linkedin.com/company/unsloth/ unslothai

| Date | Investors | Amount | Round |
| --- | --- | --- | --- |
| May 2024 | Y Combinator | $125k | Seed |
| Sep 2024 | N/A | $500k | Seed |
| Total Funding | | 000k | |

Unsloth AI develops open-source software for running and training AI models locally, including tools for reinforcement learning and fine-tuning large language models. Its desktop application supports text, image, video and audio models on Mac, Windows and Linux, with no-code training and media-generation work


### continue Continue Dev funding total valuation 2026
0 Continue - 2026 Funding Rounds & List of Investors - Tracxn https://tracxn.com/d/companies/continue/__D1vob9XV7AUB6EagrKbqSVJXJGkzXS8L6h7znRkq_bo/funding-and-investors None 
 Continue - 2026 Funding Rounds & List of Investors - Tracxn

# Continue - Funding & Investors

Last updated: October 20, 2026

## Continue's Funding Rounds

Continue has raised a total of $5.6M over 3 funding rounds: 3 Seed round. Continue's largest funding round so far was a Seed round for $***** in Feb 2025.

Here is the list of all funding rounds of Continue:

| Date of Funding | Funding Amount | Round Name | Post-Money Valuation | Revenue Multiple | Lead Investors | Other Investors |
| --- | --- | --- | --- | --- | --- | --- |
| Feb 26, 2025 | 5942796 | Seed | 7490978 | 3033565 | 5574034 | 8993574 |
| Nov 17, 2023 | 4915738 | Angel | 9211795 | 3116967 | 7633074 | Jesse Robbins & 3 more |
| 2023 | 2435 | Seed | 8560 | 8456 | Y Combinator | 9504 |

Contin


### penpot Kaleidos Penpot total funding raised 2026
0 Kaleidos - SETT https://sett.gob.es/empresas-participadas/kaleidos/ None 
 Kaleidos - SETT

# Kaleidos

### Tecnología de código abierto

La SETT ha invertido 1,92 millones de euros en la empresa española Kaleidos para impulsar su plataforma de diseño y desarrollo de interfaces de código abierto, Penpot. 

- Esta operación, que se realiza en colaboración público-privada dentro de una ronda de 6,9 millones de euros, permitirá a España y Europa avanzar en soberanía tecnológica en un campo con enorme dependencia de potencias extranjeras.
- Penpot surge de uno de los numerosos hackatones que organiza Kaleidos entre sus empleados. Se trata de una innovación que surge de las ideas de sus trabajadores, que cada seis meses se reúnen en jornadas específicas que dan alas a la creatividad de todo el equipo, una plantilla joven que cuenta con un 60% de mujeres.
- Con la aportación de la SETT, Kaleidos se asegura poder escalar el equipo comerc

In [12]:
for label in ['tiny','penpot','crew','continue','unsloth']:
 print('\n###',label)
 for i,x in enumerate(searches[label]['results']):print(i,x['url'],(x.get('full_text') or x.get('snippet') or '')[:1700])


### tiny
0 https://geohot.github.io/blog/jekyll/update/2023/05/24/the-tiny-corp-raised-5M.html the tiny corp raised $5.1M | the singularity is nearer

# the tiny corp raised $5.1M

 May 24, 2023 

Here we go again. I started another company. The money is in the bank.

### What is the tiny corp?

The tiny corp is a computer company. We sell computers for more than they cost to make; I’ve been thinking about this one for a while. In the limit, it’s a chip company, but there’s a lot of intermediates along the way.

The human brain has about 20 PFLOPS of compute. I’ve written various blog posts about this. Sadly, 20 PFLOPS of compute is not accessible to most people, costing about $1M to buy or $100/hr to rent.

With the way AI is going, we risk large entities controlling the majority of the compute in the world. I do not want “I think there’s a world market for maybe five computers.” to ever be the world we live in.

The goal of the tiny corp is: “to commoditize the petaflop”

### What i

In [13]:
for label in ['penpot','crew','continue','unsloth']:
 print('\n###',label)
 for i,x in enumerate(searches[label]['results']):print(i,x['url'],(x.get('full_text') or '')[:700])


### penpot
0 https://sett.gob.es/empresas-participadas/kaleidos/ Kaleidos - SETT

# Kaleidos

### Tecnología de código abierto

La SETT ha invertido 1,92 millones de euros en la empresa española Kaleidos para impulsar su plataforma de diseño y desarrollo de interfaces de código abierto, Penpot. 

- Esta operación, que se realiza en colaboración público-privada dentro de una ronda de 6,9 millones de euros, permitirá a España y Europa avanzar en soberanía tecnológica en un campo con enorme dependencia de potencias extranjeras.
- Penpot surge de uno de los numerosos hackatones que organiza Kaleidos entre sus empleados. Se trata de una innovación que surge de las ideas de sus trabajadores, que cada seis meses se reúnen en jornadas específicas que dan alas a l
1 https://app.dealroom.co/news/note/spain-backs-open-source-design-tool-penpot-in-6-9m-round Spain backs open-source design tool Penpot in €6.9M round | Dealroom.co

# Spain backs open-source design tool Penpot in €6.9M round

● Jul 

In [14]:
# Compact search display for subsequent batches.
async def search(label,query,n=5):
 r=await web_search(query,num_results=n); searches[label]={'query':query,'results':r}
 Path('audit_work/searches.json').write_text(json.dumps(searches,indent=2,default=str))
 print('\n###',label)
 for i,x in enumerate(r): print(i,x['title'],x['url'],'\n',(x.get('full_text') or x.get('snippet') or '')[:1300])
 return r
r=await asyncio.gather(search('tiny2','site.tinygrad.org tiny corp funding investors 2026'),search('penpot2','Penpot raises 12 million 2023 8 million funding 2022'),search('crew2','CrewAI raises 20 million April 2026 valuation total'),search('continue2','Continue Dev raises funding 2026 Heavybit valuation'),search('unsloth2','Unsloth funding seed Lightspeed 2026 raised million'))


### continue2
0 Continue - 2026 Company Profile, Team, Funding & Competitors - Tracxn https://tracxn.com/d/companies/continue/__D1vob9XV7AUB6EagrKbqSVJXJGkzXS8L6h7znRkq_bo 
 Continue - 2026 Company Profile, Team, Funding & Competitors - Tracxn

# Continue - Company Profile

Last updated: October 20, 2026

Continue is an acquired company based in San Francisco (United States), founded in 2023 by Ty Dunn. It operates as a Developer of AI agents that suggest code improvements on pull requests. Continue has raised $5.6M in funding from Y Combinator and Heavybit, with last known valuation of $*****. The company has 233 active competitors, including 12 funded and 3 that have exited. Its top competitors include companies like Verdent, Kite and Zevo.ai. 

### Company Details

Developer of AI agents that suggest code improvements on pull requests. The agents automatically suggest improvements when a pull request opens. Users can accept or reject the suggested changes. The system applies define


### penpot2
0 Penpot, the open source platform for designers and their coders, draws up $12M as users jump to 250K | TechCrunch https://techcrunch.com/2023/02/02/penpot-the-open-source-platform-for-designers-and-their-coders-draws-up-12m-as-users-jump-to-250k/ 
 Penpot, the open source platform for designers and their coders, draws up $12M as users jump to 250K | TechCrunch

Image Credits: Penpot (opens in a new window) under a (opens in a new window) license.

# Penpot, the open source platform for designers and their coders, draws up $12M as users jump to 250K

 Ingrid Lunden 

5:08 AM PST · February 2, 2023

Open source projects continue to stand out as compelling alternatives to proprietary software for those who are looking for more flexibility or price leverage in the tools that they use to work. Today one of the juggernauts in the space of design is announcing some funding. Penpot, an open source platform for designers and developers to collaborate on work in a single environme


### unsloth2
0 Unsloth AI https://linkedin.com/company/unsloth 
 # Unsloth AI

Unsloth AI is a Technology, Information and Internet company. Unsloth is an open-source, no-code deskop UI for training and running open models on your local hardware. Unsloth AI employs 12 people (+61.5% YoY, +8 people), founded in 2023. Headquartered in San Francisco, California, United States. Its workforce is distributed across United States, India, Australia, and New Zealand.

## About

Making open-source AI more accessible.

## Company Details
- Industry: Technology, Information and Internet
- Type: Privately held
- Headquarters: San Francisco, United States
- Founded Year: 2023
- Homepage: unsloth.ai
- Aliases: unsloth ai
- LinkedIn: linkedin.com/company/unsloth
- Linkedin Followers: 51,949
- Emails: support@unsloth.ai

## Financials
- Total Funding: USD 540,000
  - Seed (2024-09-30)
  - Pre seed (2024-09-05): USD 500,000
  - Angel (2024-05-24)
  - Grant (2024-05-23): USD 40,000

## Workforce
- Emplo

In [15]:
for label in ['tiny2','penpot2','crew2']:
 print('\n###',label)
 for i,x in enumerate(searches[label]['results']):print(i,x['url'],(x.get('full_text') or '')[:2300])


### tiny2
0 https://geohot.github.io/blog/jekyll/update/2023/05/24/the-tiny-corp-raised-5M.html the tiny corp raised $5.1M | the singularity is nearer

# the tiny corp raised $5.1M

 May 24, 2023 

Here we go again. I started another company. The money is in the bank.

### What is the tiny corp?

The tiny corp is a computer company. We sell computers for more than they cost to make; I’ve been thinking about this one for a while. In the limit, it’s a chip company, but there’s a lot of intermediates along the way.

The human brain has about 20 PFLOPS of compute. I’ve written various blog posts about this. Sadly, 20 PFLOPS of compute is not accessible to most people, costing about $1M to buy or $100/hr to rent.

With the way AI is going, we risk large entities controlling the majority of the compute in the world. I do not want “I think there’s a world market for maybe five computers.” to ever be the world we live in.

The goal of the tiny corp is: “to commoditize the petaflop”

### What 

In [16]:
for label in ['penpot2','crew2']:
 print('\n',label)
 for i,x in enumerate(searches[label]['results']):print(i,x['url'],(x.get('full_text') or '')[:1200])
print('continue first');print([(i,x['url'],(x.get('full_text') or '')[:1000]) for i,x in enumerate(searches['continue']['results'])])


 penpot2
0 https://techcrunch.com/2023/02/02/penpot-the-open-source-platform-for-designers-and-their-coders-draws-up-12m-as-users-jump-to-250k/ Penpot, the open source platform for designers and their coders, draws up $12M as users jump to 250K | TechCrunch

Image Credits: Penpot (opens in a new window) under a (opens in a new window) license.

# Penpot, the open source platform for designers and their coders, draws up $12M as users jump to 250K

 Ingrid Lunden 

5:08 AM PST · February 2, 2023

Open source projects continue to stand out as compelling alternatives to proprietary software for those who are looking for more flexibility or price leverage in the tools that they use to work. Today one of the juggernauts in the space of design is announcing some funding. Penpot, an open source platform for designers and developers to collaborate on work in a single environment, has raised $12 million, equity funding that it will use to drive more adoption after hitting 250,000 users this mon

In [17]:
r=await asyncio.gather(search('continue3','site.continue.dev Cursor joining acquired 2026'),search('crew3','site.crewai.com "funding" "million"'),search('penpot3','Penpot Kaleidos 2026 "total" "million" funding'),search('qdrant','Qdrant raises 2026 total funding valuation'),search('coder','Coder raises 2026 total funding valuation'))


### continue3
0 Continue (acquired by Cursor) https://continue.dev/ 
 Continue (acquired by Cursor)

# Continue has joined Cursor

Continue was acquired by Cursor. Our mission was always to ensure developers are amplified, not automated, and that commitment carries on in the work ahead.

It was an honor to build with the Continue community. Thank you to each and every one of you who helped us create a pioneering open-source coding agent.

What we built together pushed the boundaries of what AI developer tooling could be, and our open-source codebase remains freely available as a foundation for others.

## FAQs
1 Cursor quietly acquires Continue, an open-source alternative to GitHub Copilot - The New Stack https://thenewstack.io/cursor-acquires-continue-coding/ 
 Cursor quietly acquires Continue, an open-source alternative to GitHub Copilot - The New Stack

 Jul 14th 2026 2:53pm, by Alex Wilhelm 

 Jul 1st 2026 4:10pm, by Amanda Caswell 

 Nov 18th 2026 10:00am, by Cynthia Dunlop 

 No


### crew3
0 How CrewAI is orchestrating the next generation of AI Agents | Insight Partners https://www.insightpartners.com/ideas/crewai-scaleup-ai-story/ 
 How CrewAI is orchestrating the next generation of AI Agents | Insight Partners

 

# How CrewAI is orchestrating the next generation of AI Agents

Insight Partners | December 10, 2025 | 3 min. read

When João Moura started building AI Agents, he wasn’t trying to start a business — he was just trying to make his job easier.

He began experimenting during his tenure as director of AI engineering at marketing data platform, Clearbit. “I was trying to build Agents myself,” he recalls. “I was struggling with it, because back then, there were not that many options.”

Moura’s first prototype was more successful than expected: he built a group of AI Agents to automate content creation — specifically to help him post more often on LinkedIn — which resulted in up to 600 new inbound leads a day.

Instead of relying on a single AI tool, mult

In [18]:
for label in ['crew3','qdrant','coder']:
 print('\n###',label)
 for i,x in enumerate(searches[label]['results']):print(i,x['url'],(x.get('full_text') or '')[:2100])


### crew3
0 https://www.insightpartners.com/ideas/crewai-scaleup-ai-story/ How CrewAI is orchestrating the next generation of AI Agents | Insight Partners

 

# How CrewAI is orchestrating the next generation of AI Agents

Insight Partners | December 10, 2025 | 3 min. read

When João Moura started building AI Agents, he wasn’t trying to start a business — he was just trying to make his job easier.

He began experimenting during his tenure as director of AI engineering at marketing data platform, Clearbit. “I was trying to build Agents myself,” he recalls. “I was struggling with it, because back then, there were not that many options.”

Moura’s first prototype was more successful than expected: he built a group of AI Agents to automate content creation — specifically to help him post more often on LinkedIn — which resulted in up to 600 new inbound leads a day.

Instead of relying on a single AI tool, multi-Agent systems divide big tasks into smaller sub-tasks, assigning each to a speci

In [19]:
for label in ['qdrant','coder','crew2']:
 print('\n###',label)
 for i,x in enumerate(searches[label]['results']):print(i,x['url'],(x.get('full_text') or '')[:900])


### qdrant
0 https://qdrant.tech/blog/series-b-announcement/ > Explore Qdrant's agent skills catalog at https://skills.qdrant.tech/
> Search the documentation at https://search.qdrant.tech/md/documentation?q=your+query+here
> Use this file for an index of the main pages: https://qdrant.tech/llms.txt

# We Raised $50M to Build Composable Vector Search as Core Infrastructure

By Andre Zayarni, March 12, 2026

Today we're announcing $50 million in Series B funding, led by AVP, with participation from Bosch Ventures, Unusual Ventures, Spark Capital, and 42CAP.

### Retrieval Is on the Critical Path of Every AI System

Every serious AI workload — RAG, agents, multimodal search — depends on retrieving the right information, at the right time, under real constraints. Teams prototype with whatever is convenient, then hit walls in production: indexes that stall under writes, filtering applied after search instead of during it, tail latencies that spike und
1 https://www.cbinsights.com/company/

In [20]:
async def get(label,url):
 text=await fetch_parsed_page(url);pages[label]={'url':url,'text':text};Path('audit_work/pages.json').write_text(json.dumps(pages,indent=2))
 return text
fetch_jobs={
 'tiny':searches['tiny']['results'][0]['url'],
 'penpot2023':searches['penpot2']['results'][0]['url'],
 'penpot2026':searches['penpot']['results'][0]['url'],
 'crew2024':searches['crew3']['results'][2]['url'],
 'continue_exit':searches['continue3']['results'][1]['url'],
 'qdrant2026':searches['qdrant']['results'][0]['url'],
 'coder2026':searches['coder']['results'][1]['url']}
res=await asyncio.gather(*(get(k,u) for k,u in fetch_jobs.items()),return_exceptions=True)
import re
for k,t in zip(fetch_jobs,res):
 print('\n###',k)
 if isinstance(t,Exception):print(str(t));continue
 matches=[m for m in re.finditer(r'(?i).{0,180}(?:\$|million|valuation|funding|acqui|2026|2023).{0,300}',t)]
 print('\n'.join(m.group(0) for m in matches)[-11000:])


### tiny
the tiny corp raised $5.1M | the singularity is nearer
# the tiny corp raised $5.1M
May 24, 2023
jekyll/update/2020/08/20/a-silicon-person.html) [blog](https://geohot.github.io/blog/jekyll/update/2022/02/17/brain-flops.html) [posts](https://geohot.github.io/blog/jekyll/update/2023/04/26/a-person-of-compute.html) about this. Sadly, 20 PFLOPS of compute is not accessible to most people, costing about $1M to buy or $100/hr to rent.
jgnrjxsqw). Many of them managed to tape out chips, some of those chips even worked. But not a single one wrote a decent framework to use those chips. They had similar performance/$ to NVIDIA, and way worse software. Of course they failed. Everyone just bought stuff from NVIDIA.
There’s a [great chip](https://www.newegg.com/xfx-radeon-rx-7900-xtx-rx-79xmercb9/p/N82E16814150878) already on the market. For $999, you get a 123 TFLOP card with 24 GB of 960 GB/s RAM. This is the best FLOPS per dollar today, and yet…nobody in ML uses it.
Forget all that sof

2026 9:00am, by  Paul Sawers
Nov 4th 2026 1:04pm, by  Amanda Caswell
Oct 31st 2026 10:30am, by  Amanda Caswell
Oct 29th 2026 10:00am, by  Sabari Sawant and Swapnil Panchal
Oct 9th 2026 9:00am, by  Anirudh Aithal
Nov 12th 2026 10:00am, by  Ory Team
Nov 11th 2026 11:00am, by  Rak Siva
Nov 5th 2026 12:00pm, by  João Freitas
Nov 4th 2026 12:53pm, by  Matthew Burns
Oct 20th 2026 7:13am, by  Charles Humble
Nov 11th 2026 1:00pm, by  Sabari Sawant, Min Xia and Dot Ho
Nov 6th 2026 6:49am, by  Paul Sawers
Nov 5th 2026 12:14pm, by  Meredith Shubel
Nov 5th 2026 11:00am, by  Lachlan Evenson
Oct 30th 2026 3:56pm, by  Chris J. Preimesberger
Nov 11th 2026 1:00pm, by  Sabari Sawant, Min Xia and Dot Ho
Nov 5th 2026 12:14pm, by  Meredith Shubel
Nov 5th 2026 10:00am, by  Kayode Ojo
Nov 4th 2026 12:53pm, by  Matthew Burns
Oct 30th 2026 3:56pm, by  Chris J. Preimesberger
Nov 4th 2026 12:53pm, by  Matthew Burns
Nov 2nd 2026 2:03pm, by  Amanda Caswell
Oct 29th 2026 7:00am, by  Amanda Caswell
Oct 19th 2026 6:0

In [21]:
for k in ['penpot2023','crew2024','continue_exit','penpot2026']:
 t=pages[k]['text'];print('\n###',k)
 if k=='continue_exit':
  a=t.find('# Cursor quietly');print(t[a:a+12000])
 elif k=='crew2024':
  print('\n'.join(m.group(0) for m in re.finditer(r'.{0,220}(?:\$|fund|valu).{0,260}',t)))
 else:print(t[:8500])


### penpot2023
Penpot, the open source platform for designers and their coders, draws up $12M as users jump to 250K | TechCrunch

[![](https://techcrunch.com/wp-content/uploads/2026/05/tc-lockup-hp.svg "width=149 | height=16") TechCrunch Desktop Logo](https://techcrunch.com)

[![](https://techcrunch.com/wp-content/uploads/2024/09/tc-logo-mobile.svg "width=47 | height=24") TechCrunch Mobile Logo](https://techcrunch.com)

* [Latest](https://techcrunch.com/latest/)
* [Startups](https://techcrunch.com/category/startups/)
* [Venture](https://techcrunch.com/category/venture/)
* [Apple](https://techcrunch.com/tag/apple/)
* [Security](https://techcrunch.com/category/security/)
* [AI](https://techcrunch.com/category/artificial-intelligence/)
* [Apps](https://techcrunch.com/category/apps/)
* [Disrupt 2026](https://techcrunch.com/events/techcrunch-disrupt/)

* [Events](https://techcrunch.com/events/)
* [Podcasts](https://techcrunch.com/podcasts/)
* [Newsletters](https://techcrunch.com/newsletter

In [22]:
for k in ['crew2024','continue_exit']:
 t=pages[k]['text'];print('\n###',k)
 if k=='continue_exit':
  p=t.find('# Cursor quietly');t=t[p:p+12000]
 print('\n'.join(m.group(0) for m in re.finditer(r'.{0,180}(?:\$|fund|valu|Jun|acqui).{0,200}',t)))
r=await asyncio.gather(search('penpot4','site.penpot.app "funding" "12" "2023"'),search('crew4','CrewAI "Series B" "20" funding 2026 announcement'),search('qdrant2','Qdrant 2026 series B valuation undisclosed 87.8'),search('coder2','Coder "90" "total" "2026" "funding"'))


### crew2024
[Fundraising](https://techcrunch.com/category/fundraising/)
[Fundraising](https://techcrunch.com/category/fundraising/)
After a year, HubSpot acquired Clearbit, and Moura had the itch to go it alone. He’d founded startups before, including Urdog, which sold a smart collar for pets. But this go-round, Moura had a more technically ambitious concep
Investors seem to agree. CrewAI has raised $18 million across seed and Series A rounds from backers, including Boldstart Ventures, Craft Ventures, Earl Grey Capital, and Insight Partners. Coursera co-founder and AI entrepreneur Andrew Ng has al
agents-for-the-enterprise/), [Bardeen](https://techcrunch.com/2024/08/08/ai-business-agent-startup-bardeen-pulls-in-strategic-investment-from-dropbox-and-hubspot/) (which also has funding from HubSpot), [Tektonic](https://techcrunch.com/2024/06/06/tektonic-ai-raises-10m-to-build-genai-agents-for-automating-business-operations/), [11x.ai](https://techcrunch.com/2024/09/16/ai-digital
To its c


### qdrant2
0 Qdrant raises $50M Series B | Europe Alternatives | Europe Alternatives https://europealternatives.com/en/content/funding-qdrant-2026-03/ 
 Qdrant raises $50M Series B | Europe Alternatives | Europe Alternatives

2026-03-12

# Qdrant raises $50M Series B

Berlin-based Qdrant raised $50M in a Series B led by AVP.

## Round at a Glance

| Company | Qdrant |
| --- | --- |
| HQ | Berlin, Germany |
| Amount | $50M |
| Round | Series B |
| Date | 2026-03-12 |
| Valuation | Not disclosed |
| Total Disclosed Funding | $87.8M |

## Company Snapshot

Qdrant is a Berlin-based open-source vector search engine built in Rust for production AI systems. The platform supports hybrid dense-sparse search, metadata filtering, and multi-vector representations. According to the company, the project has over 25,000 GitHub stars and a community of more than 60,000 members.

## Investors in This Round

| Investor | Investor HQ Country | Notes |
| --- | --- | --- |
| AVP | United States | Lead |



### penpot4
0 Penpot, the open source platform for designers and their coders, draws up $12M as users jump to 250K | TechCrunch https://techcrunch.com/2023/02/02/penpot-the-open-source-platform-for-designers-and-their-coders-draws-up-12m-as-users-jump-to-250k/ 
 Penpot, the open source platform for designers and their coders, draws up $12M as users jump to 250K | TechCrunch

Image Credits: Penpot (opens in a new window) under a (opens in a new window) license.

# Penpot, the open source platform for designers and their coders, draws up $12M as users jump to 250K

 Ingrid Lunden 

5:08 AM PST · February 2, 2023

Open source projects continue to stand out as compelling alternatives to proprietary software for those who are looking for more flexibility or price leverage in the tools that they use to work. Today one of the juggernauts in the space of design is announcing some funding. Penpot, an open source platform for designers and developers to collaborate on work in a single environme


### coder2
0 Coder — Financials https://exa.ai/library/organization/81lkmrknd0m/financials 
 # Coder — Financials

Coder's reported financials: revenue and funding.

## Revenue

- Annual revenue: USD 5,900,000

## Funding

- Total funding: USD 172,803,428
- Rounds: 5

| Round | Announced | Amount | Valuation | Investors |
| --- | --- | --- | --- | --- |
| Series C | 2026-04-01 | USD 90,000,000 | — | Qube Research & Technologies, KKR and Uncork Capital |
| Series B | 2024-06-25 | USD 35,000,000 | — | Notable Capital, Redpoint, Georgian and Uncork Capital |
| Series B | 2020-04-15 | USD 31,823,428 | — | Notable Capital, Redpoint and Uncork Capital |
| Series A | 2019-05-02 | USD 11,480,000 | — | Notable Capital, Redpoint and Bessemer Venture Partners |
| Seed | 2018-11-08 | USD 4,500,000 | — | Capital Factory, Redpoint, Founders Fund and Uncork Capital |

1 Report: Coder Business Breakdown & Founding Story | Contrary Research https://research.contrary.com/report/coder 
 Report: Coder Bu

In [23]:
for label in ['penpot4','crew4','qdrant2','coder2']:
 print('\n###',label)
 for i,x in enumerate(searches[label]['results']):print(i,x['url'],(x.get('full_text') or '')[:1700])


### penpot4
0 https://techcrunch.com/2023/02/02/penpot-the-open-source-platform-for-designers-and-their-coders-draws-up-12m-as-users-jump-to-250k/ Penpot, the open source platform for designers and their coders, draws up $12M as users jump to 250K | TechCrunch

Image Credits: Penpot (opens in a new window) under a (opens in a new window) license.

# Penpot, the open source platform for designers and their coders, draws up $12M as users jump to 250K

 Ingrid Lunden 

5:08 AM PST · February 2, 2023

Open source projects continue to stand out as compelling alternatives to proprietary software for those who are looking for more flexibility or price leverage in the tools that they use to work. Today one of the juggernauts in the space of design is announcing some funding. Penpot, an open source platform for designers and developers to collaborate on work in a single environment, has raised $12 million, equity funding that it will use to drive more adoption after hitting 250,000 users this 

In [24]:
r=await asyncio.gather(search('tiledb','TileDB 2026 funding acquisition total valuation'),search('deepseek','DeepSeek latest funding valuation 2026 November'),search('portainer','Portainer funding 2026 total raised acquisition'),search('strapi','Strapi funding 2026 total raised valuation'),search('tiger','Tiger Data Timescale funding valuation 2026 total'))


### strapi
0 Strapi Stock Price, Funding, Valuation, Revenue & Financial Statements https://www.cbinsights.com/company/strapi/financials 
 Strapi Stock Price, Funding, Valuation, Revenue & Financial Statements

# Strapi

strapi.io

## Total Raised

$49.76M

## Investors Count

26

## Funding, Valuation & Revenue

6 Fundings

Strapi has raised $49.76M over 6 rounds.

Strapi's latest funding round was a Unattributed VC for $1.27M on January 3, 2024.

| Date | Round | Amount | Investors | Valuation Valuations are submitted by companies, mined from state filings or news, provided by VentureSource, or based on a comparables valuation model. | Revenue | Sources |
| --- | --- | --- | --- | --- | --- | --- |
| 1/3/2024 | Unattributed VC | $1.27M | Salica Investments | $XXM | 0 FY undefined | |
| 6/22/2022 | Series B | $XXM | Subscribe to see more | $XXM | 0 FY undefined | 10 |
| 9/29/2020 | Loan | | Subscribe to see more | $XXM | 0 FY undefined | 10 |
| 5/20/2020 | Series A | $XXM | Subscribe


### portainer
0 Portainer.io — Financials https://exa.ai/library/organization/9qtlz4zxc44/financials 
 # Portainer.io — Financials

Portainer.io's reported financials: revenue and funding.

## Revenue

- Annual revenue: USD 4,000,000

## Funding

- Total funding: USD 13,400,000
- Rounds: 3

| Round | Announced | Amount | Valuation | Investors |
| --- | --- | --- | --- | --- |
| Series A | 2022-06-22 | USD 6,200,000 | — | Alt Ventures, Movac, Bessemer Venture Partners, NZGCP, Shasta Ventures and Bright Pixel Capital |
| Series A | 2021-05-04 | USD 6,000,000 | — | Movac, Bessemer Venture Partners and Bright Pixel Capital |
| Seed | 2020-08-18 | USD 1,200,000 | — | K1W1, Alt Ventures, Bessemer Venture Partners and Black Nova Venture Capital |

1 Portainer.io Company Overview, Contact Details & Competitors | LeadIQ https://leadiq.com/c/portainerio/5e7f5a311d6f06b3829e6228 
 Portainer.io Company Overview, Contact Details & Competitors | LeadIQ

## Insights

Enterprise container control Por

In [25]:
for label in ['tiledb','deepseek','tiger']:
 print('\n###',label)
 for i,x in enumerate(searches[label]['results']):print(i,x['url'],(x.get('full_text') or '')[:2200])


### tiledb
0 https://www.cbinsights.com/company/tiledb/financials Tile.ai Stock Price, Funding, Valuation, Revenue & Financial Statements

# Tile.ai

## Total Raised

$54.07M

## Investors Count

21

## Funding, Valuation & Revenue

10 Fundings

Tile.ai has raised $54.07M over 10 rounds.

Tile.ai's latest funding round was a Unattributed VC for on September 16, 2026.

| Date | Round | Amount | Investors | Valuation Valuations are submitted by companies, mined from state filings or news, provided by VentureSource, or based on a comparables valuation model. | Revenue | Sources |
| --- | --- | --- | --- | --- | --- | --- |
| 9/16/2026 | Unattributed VC | | Two Bear Capital, and Amgen Ventures | $XXM | 0 FY undefined | 2 |
| 9/20/2023 | Series B - IV | $34M | AlleyCorp, Amgen Ventures, Big Pi Ventures, Endeavor, Formic Ventures, Intel Capital, LDV Partners, Lockheed Martin, Lockheed Martin Ventures, NTT DoCoMo Ventures, Nexus Venture Partners, S Ventures, Scale Asia Ventures, Two Bear Cap

In [26]:
for label in ['deepseek','tiger','portainer','tiledb']:
 print('\n###',label)
 for i,x in enumerate(searches[label]['results']):print(i,x['url'],(x.get('full_text') or '')[:1000])



### deepseek
0 https://finance.sina.com.cn/roll/2026-11-20/doc-inizkuqx5465234.shtml DeepSeek完成工商变更，千亿级融资落地投后估值6155亿_新浪财经_新浪网

# DeepSeek完成工商变更，千亿级融资落地投后估值6155亿

DeepSeek完成工商变更，千亿级融资落地投后估值6155亿

 2026年11月20日 18:34 时代周报

本文来源：时代周报 

11月20日，杭州深度求索人工智能基础技术研究有限公司完成工商变更，新增股东杭州程砚企业管理咨询合伙企业（有限合伙），注册资本由1644.7496万元增至1991.8707万元。其第二轮融资规模约1055亿元，投后估值约6155亿元，引入31家涵盖产业、资本、国资背景的合伙人，创始人梁文锋通过持股平台保留绝对控制权，已聘请中信证券筹备科创板IPO。
1 https://www.c114.com.cn/ai/5339/a1320237.html 传DeepSeek二轮融资敲定：中国移动投资80亿元 - Cloud&AI — C114通信网

2026/11/2 11:21

# 传DeepSeek二轮融资敲定：中国移动投资80亿元

C114通信网 南山

C114讯 11月2日消息（南山）日前，据美国科技媒体The Information援引消息，DeepSeek第二轮融资规模扩大至1055亿元，约30家投资方参与，公司估值达到6155亿元。

今年6月底，DeepSeek完成首轮外部融资，规模超过500亿元，投后估值约3500亿元。

据悉，DeepSeek在本轮融资中对投资方提出了明确的资源协同要求，要求所有意向投资方必须书面说明能够为DeepSeek提供何种数据支持与业务协同。

报道显示，本轮融资中，国家人工智能产业投资基金本轮出资10亿元，与首轮持平；中国移动、中国电信、中金公司、中信证券、中国人寿等机构单家出资金额在20亿至80亿元之间。

腾讯与宁德时代仍是本轮最大的产业资本。腾讯两轮均出资100亿元，累计200亿元；宁德时代体系本轮出资100亿元，较首轮的50亿元翻倍。

报道显示，运营商中，中国电信投资了“数十亿元”；中国移动则一次性投资80亿元。

从“自研保底座”到“投资壮生态”，运营商

In [27]:
r=await asyncio.gather(search('avalonia','AvaloniaUI funding investment 2026 raised'),search('mermaid','Mermaid Chart total funding raised 2026 acquisition'),search('openfaas','OpenFaaS Ltd funding investment bootstrapped acquired 2026'),search('rocket','Rocket.Chat funding raised 2026 total valuation'))


### mermaid
0 Mermaid Chart Stock Price, Funding, Valuation, Revenue & Financial Statements https://www.cbinsights.com/company/mermaid-chart/financials 
 Mermaid Chart Stock Price, Funding, Valuation, Revenue & Financial Statements

# Mermaid Chart

mermaid.ai

## Total Raised

$9.5M

## Investors Count

14

## Funding, Valuation & Revenue

4 Fundings

Mermaid Chart has raised $9.5M over 4 rounds.

Mermaid Chart's latest funding round was a Seed VC - II for on August 1, 2025.

| Date | Round | Amount | Investors | Valuation Valuations are submitted by companies, mined from state filings or news, provided by VentureSource, or based on a comparables valuation model. | Revenue | Sources |
| --- | --- | --- | --- | --- | --- | --- |
| 8/1/2025 | Seed VC - II | | Streamlined Ventures, and TMV | $XXM | 0 FY undefined | |
| 3/20/2024 | Seed VC | $XXM | Subscribe to see more | $XXM | 0 FY undefined | 10 |
| 7/1/2022 | Convertible Note | $XXM | Subscribe to see more | $XXM | 0 FY undefined | 0


### avalonia
0 One Year Into Three Million Dollars of Trust - Avalonia UI Blog https://avaloniaui.net/blog/year-one-devolutions 
 One Year Into Three Million Dollars of Trust - Avalonia UI Blog

June 24, 2026· 9 min read

# One Year Into Three Million Dollars of Trust

One year after Devolutions committed $3 million to Avalonia’s open-source development, we look back at what changed.

Written by

Mike James Chief Executive Officer

A year ago today, I announced that Devolutions had committed $3 million over three years to fund Avalonia's open-source development. I want to share what it actually changed, because the sponsorship didn't just fund more work. It dramatically changed the shape of our company.

For those of you unfamiliar with Devolutions, they're a Québec-based leader in remote connection management, including RDP and SSH, and Privileged Access Management (PAM). Avalonia is a core technology in their cross-platform strategy, making this more than just philanthropy. They're 


### openfaas
0 OpenFaaS - 2026 Company Profile, Team, Competitors & Financials - Tracxn https://tracxn.com/d/companies/openfaas/__r4QNMlweHNoiJHJiO6ZX28lfc4m7eGTBi5PkTJQzhVE 
 OpenFaaS - 2026 Company Profile, Team, Competitors & Financials - Tracxn

# OpenFaaS - Company Profile

Last updated: September 28, 2026

## OpenFaaS - About the company

OpenFaaS is an unfunded company based in Peterborough (United Kingdom), founded in 2016 by Alex Ellis. It operates as a Cloud based serverless deployment platform provider. OpenFaaS has not raised any funding yet. The company has 11 active competitors, including 1 that is funded. Its top competitors include companies like Hathora, Interactive Powers and Galactic Fog. 

### Company Details

Cloud based serverless deployment platform provider. It provides access to users to deploy functions and existing codes to Kubernetes. The platform provides tools that allow users to build both microservices and multi language functions. It enables user acces

In [28]:
for label in ['avalonia','rocket','openfaas','mermaid']:
 print('\n###',label)
 for i,x in enumerate(searches[label]['results']):print(i,x['url'],(x.get('full_text') or '')[:1200])


### avalonia
0 https://avaloniaui.net/blog/year-one-devolutions One Year Into Three Million Dollars of Trust - Avalonia UI Blog

June 24, 2026· 9 min read

# One Year Into Three Million Dollars of Trust

One year after Devolutions committed $3 million to Avalonia’s open-source development, we look back at what changed.

Written by

Mike James Chief Executive Officer

A year ago today, I announced that Devolutions had committed $3 million over three years to fund Avalonia's open-source development. I want to share what it actually changed, because the sponsorship didn't just fund more work. It dramatically changed the shape of our company.

For those of you unfamiliar with Devolutions, they're a Québec-based leader in remote connection management, including RDP and SSH, and Privileged Access Management (PAM). Avalonia is a core technology in their cross-platform strategy, making this more than just philanthropy. They're investing in the core infrastructure they depend on. It’s the sort

In [29]:
show('rocket',2200);print('\nOpenfaas sources',[(i,x['url']) for i,x in enumerate(searches['openfaas']['results'])])
print('\nTime',datetime.datetime.now(datetime.timezone.utc));print('Completed own',len(rows))


### rocket
0 Rocket Chat Company Profile: Funding, Financials and Investors | Airframe https://www.airframe.ai/product/rocket-chat/co-diligence None 
 Rocket Chat Company Profile: Funding, Financials and Investors | Airframe

# Rocket Chat

Rocket.Chat is an open-source team communications platform that provides real-time messaging, voice/video, and integrations with flexible deployment options (self-managed/on‑prem, air‑gapped, and vendor-hosted cloud) designed for regulated and mission-critical environments.

Est 2015| Valuation $30.9M| Raised $36.8M| 59 Ref Customers

LinkedIn 36K| Web traffic 368K| FTEs 181

Markets Productivity & Collaboration Suite

Platform Covers AI Assistant

Published Nov 10, 2026

## 01 Organization Info

Founded 2015

Headcount 181

## 02 Financial Stability

Total Raised $36.8M

Last Round Sep 2023

Valuation $31M

## 03 Company's YouTube Channel

No company YouTube channel available for this product.

## 05 Investors

Alexia Ventures · Andrew Schoen · Bo

In [30]:
show('crew4',2000)
r=await asyncio.gather(search('crew5','CrewAI 2026 funding "SineWave" "20"'),search('portainer2','site.portainer.io "funding" "2025" OR "2026" OR "2024"'),search('tiledb2','TileDB "34 million" "total" funding 2023'),search('tiger2','Timescale "181" "billion" "2022"'),search('deepseek2','DeepSeek November 2026 funding 86 billion 15 billion Reuters Bloomberg'))


### crew4
0 CrewAI https://enrichlayer.com/company/crewai-inc None 
 Explorium customers: we match your credits, up to 3x on top-ups. Ends Dec 2 Claim your credits

# CrewAI

Build a crew of AI agents today, scale everything tomorrow.

Last updated: 2026-10-29. For recent data, use our API.

## Overview

Crew AI is at the forefront of Agentic AI with its open source, multi-agent framework and cloud platform for building, managing and scaling agentic workflows across the entire organization. Founded as a private company, CrewAI is headquartered in San Francisco, California, and has about 46 employees.1 Its builder offers a no-code visual editor that exports to Python, as well as a code-first API, with role-based agents for orchestration.2 For governance, the platform provides real-time tracing of every LLM call, tool call, and memory read, RBAC and audit trails, and human-in-the-loop approval gates during execution.3

CrewAI's most recent funding was a $20 million Series B in April 202


### crew5
0 Portfolio - SineWave Ventures https://sinewave.vc/portfolio/ 
 Portfolio - SineWave Ventures

+800 100 900

# Portfolio

- Arize is the single platform built to help you accelerate development of AI apps and agents – then perfect them in production

- Deep learning AI platform for computer vision, natural language processing, and data labeling.

- Fast, accurate, and touchless security and temperature screening platform for any physical space.

- Affordable, high quality healthcare built for the 21st century.

- Electric bicycle network and sharing platform.

- Metronome is the usage-based billing platform that helps you launch products faster. Easily model your pricing today, iterate confidently tomorrow.

- End-to-end machine learning platform.

- Autonomous AI endpoint security platform.

- Anjuna Seaglass™ isolates workloads in a hardware-assisted environment that intrinsically secures data in every state to create a zero trust environment for code and data. Scroll for


### portainer2
0 MEDIA RELEASE: Portainer.io Closes US$6.2 Million Funding Round — Portainer https://www.portainer.io/blog/portainer-io-closes-us6-2-million-funding-round 
 MEDIA RELEASE: Portainer.io Closes US$6.2 Million Funding Round — Portainer

# MEDIA RELEASE: Portainer.io Closes US$6.2 Million Funding Round

June 22, 2022 · Neil Cresswell

Portainer.io today announced it has closed a US$6.2 million funding round. The round, which is an extension of its Series A that closed in May 2021, is led by Movac and backed by all the Series A investors (Sonae IM/Bright Pixel, Bessemer Venture Partners, BlackNova, AltVentures). It also adds US investor Shasta Ventures and New Zealand Growth Capital Partners to the roster as interest in the company grows.

Portainer launched in 2017 as an open source product to simplify the management of Docker and Swarm environments. In August 2020, it added support for enterprise Kubernetes, which is becoming the de facto industry standard for orchestrati

In [31]:
for label in ['crew5','portainer2','tiledb2','tiger2','deepseek2']:
 print('\n###',label)
 for i,x in enumerate(searches[label]['results']):print(i,x['url'],(x.get('full_text') or '')[:1400])


### crew5
0 https://sinewave.vc/portfolio/ Portfolio - SineWave Ventures

+800 100 900

# Portfolio

- Arize is the single platform built to help you accelerate development of AI apps and agents – then perfect them in production

- Deep learning AI platform for computer vision, natural language processing, and data labeling.

- Fast, accurate, and touchless security and temperature screening platform for any physical space.

- Affordable, high quality healthcare built for the 21st century.

- Electric bicycle network and sharing platform.

- Metronome is the usage-based billing platform that helps you launch products faster. Easily model your pricing today, iterate confidently tomorrow.

- End-to-end machine learning platform.

- Autonomous AI endpoint security platform.

- Anjuna Seaglass™ isolates workloads in a hardware-assisted environment that intrinsically secures data in every state to create a zero trust environment for code and data. Scroll for a visualization of how Anjuna S

In [32]:
for label in ['tiledb2','tiger2','portainer2']:
 print('\n###',label)
 for i,x in enumerate(searches[label]['results']):
  t=x.get('full_text') or ''
  print(i,x['url']);print('\n'.join(m.group(0) for m in re.finditer(r'.{0,100}(?:\$|total|million|202[0-6]).{0,160}',t))[:2700])


### tiledb2
0 https://www.linkedin.com/posts/alleycorp_tiledb-closes-34m-series-b-to-advance-the-activity-7117569962226900992-Gbfe
TileDB closes $34M Series B to advance the vision of the ‘modern database’ as an alternative to the ‘modern data stack’ | AlleyCorp
AlleyCorp is proud and excited to announce that we led TileDB's $34 million Series B funding! TileDB is reimagining the database by integrating multiple data modalities, runnable code, and versatile compute in a single product. This ne
TileDB closes $34M Series B to advance the vision of the ‘modern database’ as an alternative to the ‘modern data stack’ tiledb.com
1 https://dealroom.co/companies/tiledb/
A significant milestone in TileDB's history came in October 2023, when the company successfully closed a Series B funding round, raising $34 million. This funding was led by prominent investors, including AlleyCorp, Lockheed
 Headcount <100 Current team size Total funding $50–100M Disclosed equity funding 
 Market map Intellige

In [33]:
r=await asyncio.gather(search('coder3','site.coder.com "85" "funding"'),search('strapi2','site.strapi.io "31 million" "46"'),search('tiledb3','site.tile.ai "funding" "34"'),search('portainer3','Portainer "2025" "funding" "million"'),search('mermaid2','site.mermaid.ai "funding" "2025"'))


### coder3
0 Robert Whiteley, CEO of Coder:… - BUILDERS - Apple Podcasts https://podcasts.apple.com/us/podcast/robert-whiteley-ceo-of-coder-%2485-million-raised-to/id1619104989?i=1000668339081 
 Robert Whiteley, CEO of Coder:… - BUILDERS - Apple Podcasts

- September 4, 2024

# Robert Whiteley, CEO of Coder: $85 Million Raised to Power the Future of Developer Productivity

Welcome to another episode of Category Visionaries — the show that explores GTM stories from tech's most innovative B2B founders. In today's episode, we're speaking with Robert Whiteley, CEO at Coder, a cutting-edge developer tool that has raised $85 Million in funding.

Here are the most interesting points from our conversation:

- The Transition to CEO: Robert discusses his journey from CMO to CEO, emphasizing how his experience in data-driven marketing and operational roles prepared him for the top position at Coder.
- Insights from Analyst Firms: Robert provides an insider’s perspective on working with analyst f


### portainer3
0 Blog — Portainer https://www.portainer.io/blog 
 Blog — Portainer

# Blog

Practical guidance on Kubernetes, Docker, container governance, and platform operations.

### Portainer 3 is Kubernetes-first, not Kubernetes-only

November 6, 2026 →

### The Docker API Ceiling: Why Portainer 3.0's New Capabilities Target Kubernetes

September 21, 2026 →

### Portainer 3.0 is coming, and here's what it means for you

September 11, 2026 →

### Portainer 2.45 LTS is now available!

August 27, 2026 →

### How to Handle Kubernetes Policy Enforcement Across Clusters

August 19, 2026 →

### Multi Cluster Kubernetes Configuration Management Guide

August 18, 2026 →

### Kubernetes Secrets Types, Management, Best Practices & Solutions

August 17, 2026 →

### Kubernetes Security: Defense Models And Best Practices

August 16, 2026 →

### Deploying Critical Manufacturing MES with Portainer, Talos and Omni

August 14, 2026 →

### Portainer Business Now Tested and Certified for Critical Ma

In [34]:
for label in ['strapi2','tiledb3','mermaid2','coder2']:
 print('\n###',label)
 for i,x in enumerate(searches[label]['results']):
  t=x.get('full_text') or ''
  print(i,x['url']);print('\n'.join(m.group(0) for m in re.finditer(r'.{0,80}(?:\$|million|total|[Vv]alu).{0,200}',t))[:1600])


### strapi2
0 https://geo.sig.ai/compare/strapi-vs-s2
Paris open-source headless CMS with $49M funding (CRV/Nat Friedman Series B Jun 2022); 100% JavaScript/TypeScript, developer-first with Strapi Cloud managed hosting competing with Contentful and Sanity for API-first content management.
Strapi is a Paris, France-based open-source headless CMS company — backed with $49 million in total funding including a $31 million Series B in June 2022 led by CRV with Flex Capital, Index Ventures, and angel investor Nat Friedman (former GitHub CEO) — providing developers and d
1 https://innovation-village.com/tag/series-b-funding/
## $100M Series B Boosts Bluesky Amid CEO Transition
Bluesky Social PBC has announced a $100 million Series B funding round, marking a major…
## Maxwell+spark secures $15 million Series B to advance industrial electrification
-founded innovator in lithium-ion battery technology, has successfully closed a $15 million…
## Nigerian tech entrepreneur Tade Oyerinde secures $4

In [35]:
r=await asyncio.gather(search('qdrant3','Qdrant 2023 7.5 million 50 million valuation TechCrunch'),search('strapi3','Strapi 2022 31 million valuation 2026 CRV'),search('mermaid3','Mermaid Chart 2022 2 million seed Open Core Ventures 2025 funding'),search('rocket2','Rocket.Chat 2023 10 million funding 37 million'),search('portainer4','Portainer raises seed 1.2 million 6 million 2020 2021'))


### strapi3
0 Strapi Raises $31M Series B to Empower Millions Of People... https://strapi.io/blog/series-b 
 Strapi Raises $31M Series B to Empower Millions Of People...

# Strapi closes $31M Series B led by CRV

Pierre Burgy

● June 22, 2022● Updated on September 25, 2026

Today, we’re delighted to announce that Strapi has closed a $31M Series B round of funding! This round was led by CRV with participation from Flex Capital, Index Ventures, BPI France, Raise Sherpas as well as new notable Angel investors: Ashley Smith (Former GitLab CMO and GitHub VP of Marketing), Nat Friedman (Former GitHub CEO), Nicolas Dessaigne (Algolia Cofounder) and Matt Biilmann Christensen (Netlify CEO).

In addition to giving us significant financial resources at a time of global uncertainty, this new funding allows us to accelerate on building a sustainable business and invest in the development of the Strapi open source community and ecosystem. Most importantly, it will enable us to deliver on our missio


### qdrant3
0 Qdrant, an open source vector database startup, wants to help AI developers leverage unstructured data | TechCrunch https://techcrunch.com/2023/04/19/qdrant-an-open-source-vector-database-startup-wants-to-help-ai-developers-leverage-unstructured-data/ 
 Qdrant, an open source vector database startup, wants to help AI developers leverage unstructured data | TechCrunch

Image Credits: Qdrant. Founders — left to right: Fabrizio Schmidt, VP of Cloud; Andrey Vasnetsov, CTO; and Andre Zayarni, CEO /

# Qdrant, an open source vector database startup, wants to help AI developers leverage unstructured data

 Paul Sawers 

8:06 AM PDT · April 19, 2023

For many, ChatGPT and the generative AI hype train signals the arrival of artificial intelligence into the mainstream. But while there’s little question of a seismic sea change these past six months in terms of public awareness, the growing demand for AI could be outpacing the infrastructure required to power the myriad use cases th


### portainer4
0 Portainer.io Closes US$1.2 Million Seed Financing — Portainer https://www.portainer.io/blog/portainer-io-closes-us1-2-million-seed-financing 
 Portainer.io Closes US$1.2 Million Seed Financing — Portainer

# Portainer.io Closes US$1.2 Million Seed Financing

August 17, 2020 · Neil Cresswell

Kiwi open source startup receives significant international backing.

Auckland, New Zealand, August, 18, 2020 -- NZ based software company, Portainer.io, today announced it has closed a US$1.2 million seed financing round. The round is backed by leading strategic investors Bessemer Venture Partners and Black Nova Group, as well as top New Zealand investors K1W1, AmpliPHI Ventures and a number of experienced senior business figures.

Portainer launched in 2017 as an open source product to radically simplify the deployment and management of Docker Swarm container-based environments. To date, Portainer has been downloaded more than two billion times and is actively used by approximat

In [36]:
for label in ['qdrant3','portainer4','mermaid3']:
 print('\n###',label)
 for i,x in enumerate(searches[label]['results']):
  print(i,x['url']);t=x.get('full_text') or '';print('\n'.join(m.group(0) for m in re.finditer(r'.{0,100}(?:\$|million|valu|total).{0,160}',t))[:2000])


### qdrant3
0 https://techcrunch.com/2023/04/19/qdrant-an-open-source-vector-database-startup-wants-to-help-ai-developers-leverage-unstructured-data/
To help bring its technology deeper into the commercial sphere, Qdrant today announced a $7.5 million seed financing from lead investor Unusual Ventures, with participation from 42cap, IBB Ventures and a handful of angel backers, including Cloudera co-founder A
mr Awadallah. This is in addition to the €2 million ($2.2 million) in pre-seed funding Qdrant raised last year.
 have been taking note, too. Just last year, a similar proposition to Qdrant called Pinecone nabbed $28 million, though Zayarni considers Qdrant’s open source foundation as a major selling point for would-be customers.
, such as Zilliz, a startup commercializing the Milvus open source vector database and which raised $60 million last year. And earlier this month Chroma secured $18 million in seed funding to grow its “AI-native” open source vector database.
That Qdrant ha

In [37]:
from e14c.subagents import check_task
for k,task in tasks.items():
 s=await check_task(task);print(k,s.status,s.runtime_minutes,s.steps)
print('files',list(Path('audit_work').glob('*/*')))

smallA success 1.3633205166666669 10


smallB success 1.35046355 9
files []


In [38]:
for k,task in tasks.items():
 s=await check_task(task);print('\n###',k,'\n',s.answer)
 print('outputs',s.output_files)


### smallA 
 ## Files Ready for Analysis

| File | Description | Format | Size |
|---|---|---|---|
| `/workspace/audit_work/smallA/evidence.md` | Supporting report documenting missing inputs; not analysis-ready financial data. | Markdown | 1,234 bytes |

**Blocked:** Neither `/workspace/finance_audit_sample.csv` nor `/workspace/finance_rules.md` exists in this environment. Searches of the workspace, accessible filesystem, and available conversation memory did not recover them. The assigned row indices therefore cannot be mapped to companies.

- `/workspace/audit_work/smallA/evidence.md` records the limitation and reproduction code. Source: local input checks; no web endpoint, dataset version, or transformations apply.

No financial findings or conflicts could be assessed. No `rows.csv` or BibTeX was created, and the root manifest was not touched.

Please provide both input files at the specified paths, or supply their contents, so the ten assigned rows can be researched without guessi


### smallB 
 ## Files Ready for Analysis

| File | Description | Format | Size |
|---|---|---|---|
| `/workspace/audit_work/smallB/rows.csv` | Header-only placeholder; **not analysis-ready** | CSV | 229 bytes |
| `/workspace/audit_work/smallB/evidence.md` | Missing-input report and reproduction check | Markdown | 1,287 bytes |

**Blocked:** Both `/workspace/finance_audit_sample.csv` and `/workspace/finance_rules.md` are missing. Searches of accessible workspace and shared locations found no copies. The assigned indices therefore cannot be mapped to companies.

Files created:
- `/workspace/audit_work/smallB/rows.csv`: Exact 18 requested columns, zero data rows. Source: user-specified schema. No financial data, version, or transformations apply.
- `/workspace/audit_work/smallB/evidence.md`: Documents missing inputs, assigned indices, locations checked, and concise code to reproduce the availability check. Source: local filesystem inspection; no external endpoint applies.

No company fin

In [39]:
inline_rules=Path('finance_rules.md').read_text()
for key,inds in groups.items():
 task_prompt=f'''Research public financial disclosures for the following companies (all details inline; do NOT try to read local files). You are independent financial auditor for study. Use web searches/fetches, company/investor announcements, SEC and reputable news. ~3-5 queries/fetches per company max. Do not use modeled aggregator valuations or estimated funding. Current date 2026-11-26. Latest updates matter, confirm operating/acquired/shutdown. Input CSV:\n{src.iloc[inds].to_csv(index=False)}\nRules:\n{inline_rules}\nOverride output schema with exact user columns: {','.join(cols)}. Notes <=25 words, numbers integer USD digits or blank, unknown valuations left blank, private status_date blank. Main agent cannot access your filesystem despite environment description, so RETURN FULL CSV IN ANSWER (fenced CSV), then each company's key status/source URLs and exact financial excerpts and conflicts in prose, and full BibTeX of used sources. Do not stop for unavailable files because all input is inline. Do not upload. Need complete rows for assigned companies. Save citations via save_citation if available, but also return metadata in answer for caller. Do not fabricate.''' 
 tasks[key]=await submit_task(task_prompt,agent_type='data-retrieval',project_id=os.environ.get('PROJECT_ID'),timeout=1800)
print(tasks)

{'smallA': 'ddd0f559-7ce3-4b14-bb21-d0beeb9bf66a', 'smallB': '08d1321c-1746-402b-bad9-7f630a43bf59'}


In [40]:
r=await asyncio.gather(get('penpot_timeline',searches['penpot2']['results'][1]['url']),get('tiledb_seriesb',searches['tiledb3']['results'][0]['url']),get('mermaid_ocv',searches['mermaid3']['results'][1]['url']),get('rocket2023',searches['rocket2']['results'][0]['url']),get('coder_report',searches['coder2']['results'][1]['url']),return_exceptions=True)
for k,t in zip(['penpot_timeline','tiledb_seriesb','mermaid_ocv','rocket2023','coder_report'],r):
 print('\n###',k)
 if isinstance(t,Exception):print(t);continue
 print('\n'.join(m.group(0) for m in re.finditer(r'.{0,110}(?:\$|fund|[Vv]alu|202[1-6]).{0,230}',t))[:6500])


### penpot_timeline
[Apr 2023](https://community.penpot.app/t/faq-penpots-journey-so-far-timeline/2961/1 "Jump to the first post")
Apr 2023
## post by carolina.portugal on Apr 26, 2023
[Apr 2023](https://community.penpot.app/t/faq-penpots-journey-so-far-timeline/2961 "Post date")
**February 2021:** Penpot’s Public Launch
**2021:** Seeding Penpot’s Early Community and Moving to Beta
Penpot’s early community is seeded, and the project receives public funding in Spain. The platform moves from **Alpha to Beta in November**, experiencing a surge in adoption and love.
**2022:** Securing Series A Investment
Penpot finds the right Series A partner in Decibel, who leads an $8M investment round. This marks the largest Series A for a Spanish open-source company to date.
**January 2023:** Penpot Is Officially Out of Beta
**February 2023:** Raising $12M Series A+
Penpot [secured $12 million](https://techcrunch.com/2023/02/02/penpot-the-open-source-platform-for-designers-and-their-coders-draws-up-1

In [41]:
for k in ['mermaid_ocv','rocket2023','coder_report']:
 t=pages[k]['text'];print('\n###',k)
 print('\n'.join(m.group(0) for m in re.finditer(r'.{0,120}(?:\$[\d,.]+|[Vv]aluation|[Tt]otal funding|[Tt]otal raised).{0,220}',t))[-8500:])
print('tiledburl',pages['tiledb_seriesb']['url'])


### mermaid_ocv


### rocket2023
Rocket.Chat Raises US$10M in Series A Bridge Funding | Graphene Ventures
# Rocket.Chat Raises US$10M in Series A Bridge Funding
The company, which had already raised US$19m in February 2021, intends to use the funds for Research & Development and commercial expansion.

### coder_report
in/josh-epstein-5783aa/) as chief revenue officer after he had grown HashiCorp’s global accounts business to more than [$175 million](https://coder.com/blog/coder-experiences-2x-revenue-growth-in-2023-strengthens-leadership-team-to-accele). HashiCorp is the developer of Terraform, the infrastructure-as-code tool that Coder’s workspaces are bu
The budget Coder competes for is moving toward AI coding. Enterprises spent [$4 billion](https://menlovc.com/perspective/2025-the-state-of-generative-ai-in-the-enterprise/) on AI coding tools in 2025, up from [$550 million](https://menlovc.com/perspective/2025-the-state-of-generative-ai-in-the-ente
5-and-beyond) in early 2024. One e

In [42]:
jobs={
 'penpot2022':searches['penpot2']['results'][2]['url'],
 'strapi_rounds':searches['strapi']['results'][3]['url'],
 'mermaid2024':searches['mermaid']['results'][2]['url'],
 'tiger2022':searches['tiger']['results'][1]['url'],
 'avalonia2026':searches['avalonia']['results'][0]['url'],
 'openfaas2023':searches['openfaas']['results'][2]['url'],
 'deepseek_bloomberg':searches['deepseek']['results'][4]['url']}
r=await asyncio.gather(*(get(k,u) for k,u in jobs.items()),return_exceptions=True)
for k,t in zip(jobs,r):
 print('\n###',k)
 if isinstance(t,Exception):print(str(t));continue
 print('\n'.join(m.group(0) for m in re.finditer(r'.{0,100}(?:\$[\d.,]+|million|billion|valuation|total|funded).{0,180}',t))[:6000])


### penpot2022
Penpot Raises $8M to Keep Designers and Developers in Sync While They Build Beautiful Products
# **Penpot Raises $8M to Keep Designers and Developers in Sync While They Build Beautiful Products**
=20220927005980&lan=en-US&anchor=Kaleidos&index=2&md5=c91dd1cfb945021007163c4e19991d40), has raised $8M to build more features for designers and developers and nurture its growing community. [Decibel](https://cts.businesswire.com/ct/CT?id=smartlink&url=https%3A%2F%2Fwww.decibel.vc%
005980&lan=en-US&anchor=Taiga&index=6&md5=34afa879d923c0cf1864bbb8fccb492b), is used by more than a million people worldwide, including companies like HP, Red Hat, and Tata. Taiga’s business model includes a generous free tier, SaaS on top of the open source project, and options for on-

### strapi_rounds
* **Series B** \- 2022: We raised a **$31m series B** round, led by **CRV**, with the participation of Index Ventures and Flex Capital.
* **Series A** \- 2020: We raised a **$10m series A** round, l

In [43]:
r=await asyncio.gather(search('tiger3','Timescale 110 million 2022 total funding 181.1 valuation billion Reuters'),search('penpot_seed','Kaleidos Penpot 2021 2.6 million Athos funding'),search('penpot2026usd','Penpot Kaleidos July 2026 6.9 million dollars funding'),search('coder4','Coder 2026 funding 175 million 382 valuation KKR'),search('crew_round','CrewAI raises Series B 2026 20 million new funding Insight'))


### crew_round
0 CrewAI 2026 Company Profile: Valuation, Funding & Investors | PitchBook https://pitchbook.com/profiles/company/590845-78 
 CrewAI 2026 Company Profile: Valuation, Funding & Investors | PitchBook

# CrewAI

This is a profile preview from the PitchBook Platform.

## CrewAI Overview 

- Year Founded
- 2024

 

- Status
- Private

- Employees
- 30

 

- Latest Deal Type
- Series B

- Latest Deal Amount
- $20M

- Investors
- 15

### CrewAI General Information

#### Description

Developer of an AI agent framework designed to simplify the building and deployment of multi-agent systems to meet the diverse needs of modern teams. The company's platform provides open-source tools, pre-built models, and community-to-share models, enabling businesses to streamline teamwork, enhance communication, and optimize workflows.

#### Contact Information

##### Website

www.crewai.com

Ownership Status Privately Held (backing)

Financing Status Venture Capital-Backed

##### Corporate Offic


### tiger3
0 Year of the Tiger: $110 million to build the future of data for developers worldwide https://www.tigerdata.com/blog/year-of-the-tiger-110-million-to-build-the-future-of-data-for-developers-worldwide 
 > TimescaleDB is now Tiger Data.

Timescale just raised $110 million in our Series C, led by Tiger Global alongside all existing investors: Benchmark, New Enterprise Associates, Redpoint Ventures, Icon Ventures, and Two Sigma Ventures. With this funding, Timescale is now valued at over $1 billion, and combined with earlier rounds (2018, 2019, 2021), has raised over $180 million to fuel its growth. In the past two years, Timescale has seen 7x community growth and 20x revenue growth, with over 500 paying customers and tens of thousands of other organizations using TimescaleDB in the community today. (And yep, the Timescale Tigers raised money from Tiger Global during the Year of the Tiger 🐯🐯🐯🦄🚀💥. We don’t believe in coincidences 😉.)

All data is time-series data.

This was our

In [44]:
for label in ['penpot_seed','penpot2026usd','coder4']:
 print('\n###',label)
 for i,x in enumerate(searches[label]['results']):
  print(i,x['url']); t=x.get('full_text') or '';print('\n'.join(m.group(0) for m in re.finditer(r'.{0,100}(?:\$|€|million|[Vv]alu|[Tt]otal).{0,160}',t))[:1800])


### penpot_seed
0 https://www.europapress.es/economia/noticia-tecnologica-kaleidos-cierra-ronda-22-millones-20210720103818.html

1 https://www.expansion.com/expansion-empleo/emprendedores/2021/07/20/60f51db9468aeb832b8b4655.html

2 https://elreferente.es/inversiones/kaleidos-cierra-una-ronda-seed-de-22-me-liderada-por-athos-capital/
Kaleidos cierra una ronda seed de 2,2 M€ liderada por Athos Capital
# Kaleidos cierra una ronda seed de 2,2 M€ liderada por Athos Capital 
3 https://venturemap.es/round/1357
# Kaleidos — Seed · €2.2M
Kaleidos closed its first institutional round, a €2.2M seed led by Madrid fund Athos Capital (chosen from 15 interested funds), with public innovation agency CDTI also contributing and returning business angels
its own products, Taiga and Penpot; at its 2011 founding as Ars Code the company had raised roughly €1M in initial capital from the founding team, nearby angels, a private investor and the Secuoyas group, which was never announced as a venture round.
4 

In [45]:
jobs={
 'deepseek_reuters':searches['deepseek2']['results'][4]['url'],
 'deepseek_caixin':searches['deepseek']['results'][2]['url'],
 'penpot_nordic':searches['penpot2026usd']['results'][3]['url'],
 'coder_info':searches['coder4']['results'][2]['url'],
 'tiger_release':searches['tiger3']['results'][4]['url']}
r=await asyncio.gather(*(get(k,u) for k,u in jobs.items()),return_exceptions=True)
for k,t in zip(jobs,r):
 print('\n###',k)
 if isinstance(t,Exception):print(t);continue
 print('\n'.join(m.group(0) for m in re.finditer(r'.{0,130}(?:\$|million|billion|valuation|total|July|Jun).{0,230}',t))[:7500])


### deepseek_reuters
DeepSeek wrapping up $15 billion fundraising to fuel AI race, sources say
# DeepSeek wrapping up $15 billion fundraising to fuel AI race, sources say
[Article](https://www.streetinsider.com/Reuters/DeepSeek+wrapping+up+%2415+billion+fundraising+to+fuel+AI+race%2C+sources+say/27217623.html)
ite.png) Tweet](https://twitter.com/share?url=http%3A%2F%2Fwww.streetinsider.com%2FReuters%2FDeepSeek%2Bwrapping%2Bup%2B%252415%2Bbillion%2Bfundraising%2Bto%2Bfuel%2BAI%2Brace%252C%2Bsources%2Bsay%2F27217623.html&via=Street_Insider&text=DeepSeek+wrapping+up+%2415+billion+fundraising+to+fuel+AI+race%2C+sources+say) [![](https://www.streetinsider.com/images/fb-white.
(https://www.facebook.com/sharer/sharer.php?u=http%3A%2F%2Fwww.streetinsider.com%2FReuters%2FDeepSeek%2Bwrapping%2Bup%2B%252415%2Bbillion%2Bfundraising%2Bto%2Bfuel%2BAI%2Brace%252C%2Bsources%2Bsay%2F27217623.html) [![](https://www.streetinsider.com/images/icon_email.gif) E-mail](https://www.streetinsider.com/mailme.ph

In [46]:
r=await asyncio.gather(search('deepseek3','DeepSeek valuation "November" "2026" "$" "billion" funding'),search('tiledb4','TileDB raises 2023 34 million total funding 54 million'),search('rocket3','Rocket.Chat 2023 10 million funding total 37 million'),search('penpot5','Kaleidos funding 2026 "30" "million"'),search('openfaas2','site.openfaas.com "bootstrapped"'))


### deepseek3
0 JMI Equity https://linkedin.com/company/jmi-equity 
 # JMI Equity

JMI Equity is a Venture Capital and Private Equity Principals company. JMI Equity is a leading partner and investor in software and AI-driven companies. They focus on backing outstanding software and AI-driven companies through capital investments, strategic advice, team-building, and proven value creation strategies. JMI Equity employs 113 people (+5.3% YoY, +8 people) and has an annual revenue of $24.4M, founded in 1992. Headquartered in Washington, District of Columbia, United States, with offices in Baltimore and La Jolla. Its workforce is distributed across 5 countries (including United States, India, Bangladesh, Lithuania, and United Kingdom).

## About

Founded in 1992, JMI Equity has focused on investing in exceptional software and AI-driven companies from scaling stages through later growth. For more than three decades, JMI has partnered with founder-owners and management teams across market cy


### openfaas2
0 As OpenFaaS continues to evolve, it’s time to update your images | OpenFaaS - Serverless Functions Made Simple https://www.openfaas.com/blog/time-to-update/ 
 As OpenFaaS continues to evolve, it’s time to update your images | OpenFaaS - Serverless Functions Made Simple

# As OpenFaaS continues to evolve, it's time to update your images 

## We've been doing some housekeeping with old container images, so it's time to update your OpenFaaS installation. 

 July 28, 2023 Openfaas Community Faasd Ce Vision 

We’ve been doing some housekeeping with old container images, so it’s time to update your OpenFaaS installation.

In this article I’ll explain how we’ve been doing housekeeping on old images for a while now, starting with the Docker Hub reset, why it’s so important to be running up to date versions of OpenFaaS, how to update your Helm chart and how the project has been evolving over the past few years of full-time work.

## Resetting the Docker Hub and the CVE firehose


### rocket3
0 Rocket.Chat — Financials https://exa.ai/library/organization/5gb29rt2tdl/financials 
 # Rocket.Chat — Financials

Rocket.Chat's reported financials: revenue and funding.

## Revenue

- Annual revenue: USD 4,000,000

## Funding

- Total funding: USD 36,900,313
- Rounds: 5

| Round | Announced | Amount | Valuation | Investors |
| --- | --- | --- | --- | --- |
| Series A | 2023-09-21 | USD 10,000,000 | — | Endeavor Catalyst, MONASHEES, bossa invest, Scale-Up Ventures, Greycroft, Valor Capital Group, Graphene Ventures and DGF |
| Series A | 2021-02-03 | USD 19,000,000 | — | MONASHEES, ONEVC, Greycroft, Valor Capital Group, e.ventures, OSS Capital, Graphene Ventures and DGF |
| Seed | 2019-08-06 | USD 2,900,313 | — | MONASHEES, bossa invest, ONEVC and DGF |
| Seed | 2018-03-01 | — | — | bossa invest |
| Seed | 2016-10-11 | USD 5,000,000 | — | — |

1 Gunderson Dettmer Advises Alexia Ventures in its $10M Series A Led Rocket.Chat | Gunderson Dettmer Stough Villeneuve Franklin & 


### tiledb4
0 TileDB closes $34M Series B to advance the vision of the ‘modern database’ as an alternative to the ‘modern data stack’ https://www.businesswire.com/news/home/20231010796451/en/TileDB-closes-%2434M-Series-B-to-advance-the-vision-of-the-modern-database-as-an-alternative-to-the-modern-data-stack 
 TileDB closes $34M Series B to advance the vision of the ‘modern database’ as an alternative to the ‘modern data stack’

TileDB closes $34M Series B to advance the vision of the ‘modern database’ as an alternative to the ‘modern data stack’

TileDB envisions evolving the ‘database’ by integrating multiple data modalities, runnable code and versatile compute in a single product

CAMBRIDGE, Mass.--(BUSINESS WIRE)-- TileDB Inc., today announced the closing of its Series B funding round, raising $34 million to advance its vision of the “modern database” as a single secure product for enterprises, developers and scientists. The funding was led by AlleyCorp, founded by Kevin Ryan (co-c


### penpot5
0 Radiant: funding, secondary venues and the acquisition read | PiggyGPT https://piggygpt.com/private/radiant 
 Radiant: funding, secondary venues and the acquisition read | PiggyGPT

# Radiant

El Segundo, CA · Founded 2020

Radiant builds Kaleidos, a 1-megawatt nuclear microreactor shipped by truck, to replace diesel generators at military bases, data centers and remote sites. The Army has an agreement for 15 units under its Janus program, and the Air Force selected it for Buckley Space Force Base.

## Profile

### What it builds and who buys it

- · Kaleidos is a 1 MWe, helium-cooled microreactor fueled with HALEU TRISO fuel. It is factory-built and arrives as a sealed, fueled unit. POWER, Aug 27, 2026
- · The company says each unit runs up to five years before refueling and has a 20-year operating life. It plans to handle refueling and spent-fuel storage away from customer sites. POWER, Aug 27, 2026
- · Kaleidos also delivers 1.9 megawatts of heat for buildings or desa

In [47]:
for label in ['tiledb4','rocket3','openfaas2']:
 print('\n###',label)
 for i,x in enumerate(searches[label]['results']):
  print(i,x['url']);t=x.get('full_text') or '';print('\n'.join(m.group(0) for m in re.finditer(r'.{0,90}(?:\$|[Tt]otal|[Bb]ootstrapped|funded|million).{0,170}',t))[:2600])


### tiledb4
0 https://www.businesswire.com/news/home/20231010796451/en/TileDB-closes-%2434M-Series-B-to-advance-the-vision-of-the-modern-database-as-an-alternative-to-the-modern-data-stack
TileDB closes $34M Series B to advance the vision of the ‘modern database’ as an alternative to the ‘modern data stack’
TileDB closes $34M Series B to advance the vision of the ‘modern database’ as an alternative to the ‘modern data stack’
S WIRE)-- TileDB Inc., today announced the closing of its Series B funding round, raising $34 million to advance its vision of the “modern database” as a single secure product for enterprises, developers and scientists. The funding was led by AlleyCorp, founded
1 https://indexed.vc/companies/tiledb
TileDB Funding History: $54.0M | Indexed.vc
tificial intelligence company in Cambridge, United States, founded in 2017. It has raised $54.0M across 4 rounds, most recently a $34.0M Series B in October 2023.
Total Funding
$54.0M
$34.0M Series B Oct 2023
Totals include re

In [48]:
t=pages['coder_info']['text'];p=t.find('# Exclusive');print(t[p:p+6500])
print('\nCoder valuation section:')
t=pages['coder_report']['text'];p=t.find('## Valuation');print(t[p:p+6000])
print('\nPenpot Nordic start:');t=pages['penpot_nordic']['text'];print(t[:4800])

# Exclusive: Coder Raises $90 Million Led by KKR

By 

[Julia Hornstein](https://www.theinformation.com/u/juliahornstein0w878c)

Get full access, explore exclusive reporting.

Continuar con GoogleContinuar con Google. Se abre en una pestaña nueva

[Download the App](https://theinformation.onelink.me/6IaG/3stnun15)

* [About](https://www.theinformation.com/about)
* [Pro](https://www.theinformation.com/pro)
* [Deep Research](https://www.theinformation.com/deep-research)
* [Careers](https://www.theinformation.com/jobs)
* [TITV](https://www.theinformation.com/titv)
* [Events](https://www.theinformation.com/events)
* [Brand Partnerships](https://www.theinformation.com/partners)
* [Group Subscriptions](https://www.theinformation.com/corporate)

Sign up for newsletters

[X](https://x.com/theinformation)[Facebook](https://www.facebook.com/gettheinformation)[LinkedIn](https://www.linkedin.com/company/theinformation)[Threads](https://www.threads.net/@theinformation)[Instagram](https://www.instag

In [49]:
jobs={
 'coder2024':'https://coder.com/newsroom/press-releases/coder-announces-35m-in-series-b2-funding-to-modernize-the-enterprise-development',
 'coder2020':'https://news.crunchbase.com/news/austin-based-coder-raises-30m-for-cloud-based-software-development/',
 'tiledb2020':'https://techcrunch.com/2020/07/14/tiledb-raises-15m-series-a-for-its-modern-database/',
 'openfaas_current':searches['openfaas2']['results'][2]['url']}
r=await asyncio.gather(*(get(k,u) for k,u in jobs.items()),return_exceptions=True)
for k,t in zip(jobs,r):
 print('\n###',k)
 if isinstance(t,Exception):print(str(t));continue
 print('\n'.join(m.group(0) for m in re.finditer(r'.{0,120}(?:\$|million|valuation|total|bootstrapped).{0,230}',t))[:6000])


### coder2024
Coder Raises $35M to Accelerate the Cloud Development Environment Market with Extended Open Source, Enhanced CDE Capabilities, and Expanded Global Reach - Blog - Coder
# Coder Raises $35M to Accelerate the Cloud Development Environment Market with Extended Open Source, Enhanced CDE Capabilities, and Expanded Global Reach
I am thrilled to announce Coder raised $35 million in fresh capital, bringing our Series B total to $65M. This marks a significant step forward in our mission to make developers 10x more productive, and underscores our commitment to empowering the global developer community. I’m extremely proud of the Coder team for hitting this 
But we can do more. We can do better. That’s why we raised another $35M. Coder is doubling down on CDEs to take the next step in our mission of making developers 10x more productive. Our goal has always been to keep developers in flow.
 a new Developer Experience function dedicated to overseeing and growing our community, which 

In [50]:
r=await asyncio.gather(search('coder5','Coder raises 35 million 2024 total funding 85 million'),search('tiledb5','TileDB raises 15 million 2020 total 20 million funding'),search('rocket4','Rocket.Chat 19 million 2021 total funding 27 million'))


### coder5
0 Coder Raises $35M to Accelerate the Cloud Development Environment Market with Extended Open Source, Enhanced CDE Capabilities, and Expanded Global Reach https://coder.com/blog/coder-raises-35m-in-latest-funding-round 
 Jun 25 20245 min read

# Coder Raises $35M to Accelerate the Cloud Development Environment Market with Extended Open Source, Enhanced CDE Capabilities, and Expanded Global Reach

Rob Whiteley

Share this article

I am thrilled to announce Coder raised $35 million in fresh capital, bringing our Series B total to $65M. This marks a significant step forward in our mission to make developers 10x more productive, and underscores our commitment to empowering the global developer community. I’m extremely proud of the Coder team for hitting this huge milestone.

## Transforming How Developers Stay in the Flow

Today’s development environments are evolving rapidly. Developers traditionally built, shipped, and ran their code in the cloud, yet relied on their local ma


### rocket4
0 Rocket.Chat Raises Series A Round | Woden: Story Is The Strategy https://storyisthestrategy.com/news/rocket-chat-raises-series-a-round/ 
 Rocket.Chat Raises Series A Round | Woden: Story Is The Strategy

# Rocket.Chat Raises Series A Round

Woden

Feb 3, 2021

Woden client Rocket.Chat announced today that it has raised a $19 million Series A funding round. The open-source chat platform, based in Brazil, closed the new round after a year when reliable online communication tools became essential to just about every business worldwide.

Since launching in 2015, Rocket.Chat has raised $27 million in total funding. It was after one of those early rounds that the platform engaged Woden to help develop a story that differentiated it from other team communication platforms.

The company offers a secure group chat app as well as communication solutions for things like customer service and conferences. Rocket.Chat’s early product was entirely open-source; Woden helped craft the me


### tiledb5
0 TileDB puts $15m on the slate https://globalventuring.com/blog/2020/07/20/tiledb-puts-15m-on-the-slate/ 
 Full Site Corporate University Impact

- News
- People
- Analysis
- Deals
- Size
- Sectors
- Region
- Comment

- News
- Deals
- Size
- Sectors
- Region
- Features

Register Login

Go

Sectors

Sectors Art of CVC Consumer Data Energy Financial Health Industrial IT Media Services Startups Telecoms Transport

- News & Analysis Home Latest News Reports Podcasts Data The CVC Funding Round Database The CVC Directory Video Subscribe Newsletters Job Postings
- Events Events The Next Wave Webinar Series
- GCV Institute Professional Development Keystone CVC Benchmarking
- Communities Communities Membership Industry Councils Subscribe The CVC Directory
- Register
- Login

Sectors Art of CVC Consumer Data Energy Financial Health Industrial IT Media Services Startups Telecoms Transport

# TileDB puts $15m on the slate

#### Jul 20, 2020 • Callum Cyrus

Intel Capital returned for 

In [51]:
for label in ['coder5','rocket4','tiledb5']:
 print('\n###',label)
 for i,x in enumerate(searches[label]['results']):
  t=x.get('full_text') or ''
  print(i,x['url']);print('\n'.join(m.group(0) for m in re.finditer(r'.{0,110}(?:\$|[Tt]otal).{0,220}',t))[:2600])


### coder5
0 https://coder.com/blog/coder-raises-35m-in-latest-funding-round
# Coder Raises $35M to Accelerate the Cloud Development Environment Market with Extended Open Source, Enhanced CDE Capabilities, and Expanded Global Reach
I am thrilled to announce Coder raised $35 million in fresh capital, bringing our Series B total to $65M. This marks a significant step forward in our mission to make developers 10x more productive, and underscores our commitment to empowering the global developer community. I’m extremely proud of the Coder team for hit
But we can do more. We can do better. That’s why we raised another $35M. Coder is doubling down on CDEs to take the next step in our mission of making developers 10x more productive. Our goal has always been to keep developers in flow.
1 https://www.dealdata.net/company-profile/0001756001/
## Total Amount Raised: 
###### $85,233,397.00
Round # Round Open Date Amount Raised Total Round Size Amount Remaining to be Raised Securities
4 2024-06-2

In [52]:
put(11,status='private',total_funding_usd='5100000',funding_asof='2023-05',funding_source=cite_result('tiny',0),ever_1b='no',notes='Founder announced $5.1m; proposed 2026 fundraising is not a completed round or valuation.')
put(12,status='private',total_funding_usd='30540000',funding_asof='2026-07',funding_source=cite_result('penpot2026usd',3),ever_1b='no',notes='USD-equivalent cumulative funding reported by Nordic9; includes July 2026 EUR6.9m round. Database totals conflict.')
cite_result('penpot',0,'Official 2026 EUR6.9m equity round corroborates Nordic9 latest event');cite_result('penpot2',1,'Company timeline confirms distinct $8m (2022) and $12m (2023) rounds')
put(13,status='private',total_funding_usd='38000000',funding_asof='2026-04',funding_source=cite_result('crew3',2)+'|'+cite_result('crew2',0),last_valuation_usd='100000000',valuation_date='2024-10',valuation_source=cite_result('crew3',2),peak_value_usd='100000000',peak_value_date='2024-10',peak_source=cite_result('crew3',2),ever_1b='no',notes='Reported approximate $100m valuation; $18m cumulative plus PitchBook $20m Series B. Later valuations and cumulative totals conflict.')
cite_result('crew_round',1,'Investor confirms Series B without amount')
put(14,status='acquired',status_date='2026-06',acquirer='Cursor (Anysphere)',exit_source=cite_result('continue3',0),total_funding_usd='5000000',funding_asof='2025-02',funding_source=cite_result('continue',3),ever_1b='no',notes='Acqui-hire; hosted product discontinued. Company press release reports $5m cumulative; databases disagree.')
cite_result('continue3',1,'Acquisition report dated June 22 says website changed around June 16; terms undisclosed')
put(15,status='private',ever_1b='no',notes='Investor-backed small team; no qualifying round amounts or valuation independently verified; grants excluded.')
cite_result('unsloth',4,'YC backing, no amount stated');cite_result('unsloth2',4,'Investor confirms private operating company')
put(16,status='private',total_funding_usd='87800000',funding_asof='2026-03',funding_source=cite_result('qdrant',2),ever_1b='unknown',notes='Latest $50m Series B confirmed by company; no reliable disclosed post-money valuation located.')
cite_result('qdrant',0,'Company confirms $50m Series B March 12 2026')
put(19,status='private',total_funding_usd='175000000',funding_asof='2026-04',funding_source=cite_result('coder3',0)+'|'+cite_result('coder',1),ever_1b='unknown',notes='Rounded CEO-reported $85m plus $90m Series C; published cumulative totals and valuations conflict.')
put(25,status='private',total_funding_usd='54000000',funding_asof='2023-10',funding_source=cite_result('tiledb5',0)+'|'+cite_result('tiledb4',0),ever_1b='unknown',notes='Rebranded Tile.ai; September 2026 strategic investment undisclosed, excluded from counted $54m. No verified post-money valuation.')
cite_result('tiledb',2,'2026 rebrand and undisclosed strategic financing')
put(26,status='private',total_funding_usd='22400000000',funding_asof='2026-11',funding_source=cite_result('deepseek2',4)+'|'+cite_result('deepseek',2),last_valuation_usd='86000000000',valuation_date='2026-11',valuation_source=cite_result('deepseek',4),peak_value_usd='86000000000',peak_value_date='2026-11',peak_source=cite_result('deepseek',4),ever_1b='yes',notes='Reported approximate USD figures: $7.4b plus $15b rounds; Bloomberg implied post-money mark. Amounts and closing dates differ between reports.')
put(27,status='private',total_funding_usd='13400000',funding_asof='2022-06',funding_source=cite_result('portainer4',2)+'|'+cite_result('portainer',3),ever_1b='no',notes='Sum of company-announced equity rounds; database-only 2025 financing not independently corroborated, excluded.')
put(29,status='private',total_funding_usd='46000000',funding_asof='2022-06',funding_source=cite_result('strapi',3),ever_1b='unknown',notes='Company equity history sums $46m; debt and unverified later secondary transactions excluded. No disclosed valuation verified.')
put(31,status='private',ever_1b='no',notes='Devolutions $3m is a three-year open-source sponsorship, not equity funding; company remains autonomous.')
cite_result('avalonia',1,'Company labels $3m funding sponsorship and preserves autonomy');cite_result('avalonia',0,'2026 update confirms continued independent operations')
put(32,status='private',total_funding_usd='7500000',funding_asof='2024-03',funding_source=cite_result('mermaid',2),ever_1b='no',notes='Only company-announced $7.5m seed counted; databases report $9.5m including earlier convertible financing; no valuation verified.')
put(33,status='private',ever_1b='no',notes='Founder describes bootstrapped business; no disclosed external equity funding or valuation located.')
cite_result('openfaas2',2,'Founder states bootstrapped company and current products')
put(38,status='private',total_funding_usd='180000000',funding_asof='2022-02',funding_source=cite_result('tiger3',4),valuation_date='2022-02',valuation_source=cite_result('tiger3',4),peak_value_date='2022-02',peak_source=cite_result('tiger3',4),ever_1b='yes',notes='Company disclosed valuation only as over $1b; exact valuation left blank. Official $180m total differs from databases.')
put(39,status='private',total_funding_usd='37000000',funding_asof='2023-09',funding_source=cite_result('rocket4',3)+'|'+cite_result('rocket2',1),ever_1b='unknown',notes='$27m cumulative plus $10m 2023 equity bridge; convertible debt excluded. Conflicting database valuations not adopted.')
cite_result('rocket',4,'Company operating page updated October 2026')
print('Saved',len(rows),'own rows.');print(pd.DataFrame(rows.values())[['entity','status','total_funding_usd','ever_1b']].to_string(index=False))

Saved 19 own rows.
            entity    status total_funding_usd ever_1b
           Anomaly   private           1000000      no
         Kilo Code  acquired           8000000      no
       OpenBB Inc. shut_down           8500000      no
         Tiny Corp   private           5100000      no
 Kaleidos (Penpot)   private          30540000      no
            CrewAI   private          38000000      no
Continue Dev, Inc.  acquired           5000000      no
        Unsloth AI   private                        no
            Qdrant   private          87800000 unknown
Coder Technologies   private         175000000 unknown
      TileDB, Inc.   private          54000000 unknown
          DeepSeek   private       22400000000     yes
      Portainer.io   private          13400000      no
            Strapi   private          46000000 unknown
     AvaloniaUI OÜ   private                        no
     Mermaid Chart   private           7500000      no
      OpenFaaS Ltd   private                  

In [53]:
for k,task in tasks.items():
 s=await check_task(task);print(k,s.status,s.runtime_minutes,s.steps); 
 if s.answer:Path(f'audit_work/{k}_agent_answer.md').write_text(s.answer);print(s.answer[:2000])
print('UTC',datetime.datetime.now(datetime.timezone.utc).isoformat())

Task ddd0f559-7ce3-4b14-bb21-d0beeb9bf66a still running (status: in progress). Call check_task(task_id) again later.
smallA in progress 3.5892345666666667 20


Task 08d1321c-1746-402b-bad9-7f630a43bf59 still running (status: in progress). Call check_task(task_id) again later.
smallB in progress 3.336096 22
UTC 2026-11-26T06:02:10.926518+00:00


In [54]:
print('Note word counts:',[(i,len(d['notes'].split())) for i,d in rows.items() if len(d['notes'].split())>25])
print('Used sources',len(used)); print('Search requests',len(searches));print('pages',len(pages))
# Save progress in the requested path now; it will be replaced with all 40 rows after integration.
pd.DataFrame([rows.get(i,{**{c:'' for c in cols},**src.iloc[i][['github_orgs','entity']].to_dict()}) for i in range(len(src))],columns=cols).to_csv('finance_audit.csv',index=False)
# Inspect only existing project conventions before any possible update.
for f in ['/workspace/.prompts/MEMORY.md','/workspace/.prompts/USER.md']:
 print(f); print(Path(f).read_text()[:3000] if Path(f).exists() else 'Absent')

Note word counts: []
Used sources 38
Search requests 63
pages 26
/workspace/.prompts/MEMORY.md
# Project Memory

## Instructions

Edit this file (`{prompt_path}`) to persist procedural and workflow patterns specific to this project: output formats, delivery channels, and recurring conventions. Do not record scientific findings or research outputs here. Update or remove entries as the project's conventions evolve.

## Context

[Fill me in]

## Notes

[Fill me in]

/workspace/.prompts/USER.md
# User

Edit this file (`{prompt_path}`) to remember stable facts about the user that should persist across turns and sessions: how collaborative they want you to be, autonomy preferences, cadence and check-in expectations, voice and tone preferences, recognized persona patterns, and any other durable preferences they state.

Keep entries concise. One bullet per preference. Update or remove entries as
the user's preferences evolve. Do not record transient task state or scientific
findings here — use

In [55]:
r=await asyncio.gather(search('anomaly_current','Anomaly OpenCode funding raised acquired November 2026 company'),search('mermaid_current','Mermaid Chart funding total 2026 seed round investor'),search('crew_valuation','CrewAI Series B 2026 valuation 185 million 177.15 million'),search('qdrant_valuation','Qdrant Series B valuation 2026 50 million undisclosed'))


### anomaly_current
0 OpenCode Company Profile: 10 Employees | TrueUp https://www.trueup.io/co/anomaly-dev 
 OpenCode Company Profile: 10 Employees | TrueUp

# OpenCode

Open-source AI coding agent (part of Anomaly Co)

## Overview

Founded

2008

19 years ago

Company type

Private - Venture-Backed

OpenCode is an open-source AI coding agent. Founded in 2008, it has about 10 employees. Its latest funding was a seed round in 2021. Investors include YC.

Updated Nov 26, 2026

## Jobs

No open jobs listed for OpenCode.

## Products

### Recent launch

O

OpenCode @opencode · Nov 23

OpenCode 2.4.0 ships with a new automation tool your agent can create automations that run on schedule or react to mcp events they can trigger prompts, shell commands, or a code mode script that can access your mcp tools

## People

Employees

10

Live Data Technologies estimate est. About Live Data estimates About Live Data estimates

## Open source

### Top projects

GitHub stars and their change over the 

In [56]:
for label in ['crew_valuation','qdrant_valuation']:
 print('\n###',label)
 for i,x in enumerate(searches[label]['results']):
  print(i,x['url']);t=x.get('full_text') or '';print('\n'.join(m.group(0) for m in re.finditer(r'.{0,100}(?:\$|[Vv]alu|[Tt]otal).{0,200}',t))[:2600])


### crew_valuation
0 https://earlgrey.capital/updates
 Fund II is performing well. We raised ~$13M and have called 44% of commitments ($5.72M) to date, building a portfolio of 14 pre-seed and seed-stage companies. 
l core positions (CrewAI at 8.86x, Rubie at 3.55x, Archetype AI at 2.96x). We've distributed nearly $1M in cash to LPs so far, with another ~$1.15M in pending distributions from locked securities and escrow over the next 12-18 months. Where it made sense, we recycled exit proceeds tax-free into new p
Benchmarks: Carta VC Fund Performance Q4 2025, 2023 vintage ($10M-$25M fund cohort)
$951K
$775K
~$1.15M
ial) - our first check in Fund II. Stablecoin payments infrastructure. Acquired by Ripple at 5.16x. $938K distributed to LPs (escrow coming) + $300K recycled. We also still hold Ripple shares with additional liquidation pending.
45. COIN shares locked until January 2027 when we'll distribute to LPs. Near-term COIN volatility (~$220K unrealized drag) is noise; we're holding to

In [57]:
# Fetch returned HTTP 401; preserve accessible search-index excerpt instead, without claiming a successful fetch.
x=searches['crew_valuation']['results'][0]
print('Full-page fetch blocked (HTTP 401). Search-index evidence:')
print(x['full_text'])
pages['crew_investor']={'url':x['url'],'text':x['full_text'],'access':'search-index excerpt; full-page fetch HTTP 401'}
Path('audit_work/pages.json').write_text(json.dumps(pages,indent=2));

Full-page fetch blocked (HTTP 401). Search-index evidence:
Fund II Update - Earl Grey Capital

# Fund II · May 2026 

 Performance, portfolio, and observations 

 Fund II is performing well. We raised ~$13M and have called 44% of commitments ($5.72M) to date, building a portfolio of 14 pre-seed and seed-stage companies. 

 We've had two exits already (Rail to Ripple at 5.16x, Clearing Company to Coinbase at 2.34x) alongside meaningful markups on several core positions (CrewAI at 8.86x, Rubie at 3.55x, Archetype AI at 2.96x). We've distributed nearly $1M in cash to LPs so far, with another ~$1.15M in pending distributions from locked securities and escrow over the next 12-18 months. Where it made sense, we recycled exit proceeds tax-free into new positions to keep that capital compounding. 

 The result: 1.80x Net TVPI, ~33.5% Net IRR, and 0.16x DPI, which puts Fund II in the top decile of 2023 vintage venture funds. Full breakdown and quarterly timeline live on the performance dashboar

In [58]:
r=await asyncio.gather(search('crew_primary','CrewAI Series A 18 million seed Insight funding October 2024'),search('crew_primary2','CrewAI seed funding 6.5 million 18 million 2024 investor'),search('mermaid_ocv_funding','Mermaid Chart initial investment 2 million Open Core Ventures'))


### mermaid_ocv_funding
0 Open Core Ventures — We build commercial open source https://opencoreventures.com/insights/why-we-invested-in-mermaid-chart/ 
 Open Core Ventures — We build commercial open source

Oct 10, 2022 · 4 min read

# Why we invested in Mermaid Chart

Erica Lindberg

Mermaid Chart is changing how we communicate. Creating dynamic text-based diagrams removes the clumsy design layer and improves collaboration.

Mermaid Chart is changing how we communicate. Creating dynamic text-based diagrams removes the clumsy design layer and improves collaboration.

Knut Sveidqvist was with his kids in the family room when he sat down and started writing Mermaid. “They were watching The Little Mermaid,” said Knut about naming the project. “I did not for one moment guess what would happen.”

Like many open source software projects, Knut created Mermaid to address a problem he was experiencing: bad documentation. During nights and weekends, he worked on making diagrams more fun and eff


### crew_primary2
0 Agentic AI startup CrewAI closes $18M funding round - SiliconANGLE https://siliconangle.com/2024/10/22/agentic-ai-startup-crewai-closes-18m-funding-round/ 
 Agentic AI startup CrewAI closes $18M funding round - SiliconANGLE

 UPDATED 19:14 EDT / OCTOBER 22 2024 

### Agentic AI startup CrewAI closes $18M funding round

 by Maria Deutscher 

CrewAI Inc., the developer of a popular tool for building artificial intelligence agents, has raised $18 million in funding to support its growth efforts.

The startup announced the milestone today. It raised the capital over two rounds: a seed investment led by boldstart ventures and a Series A raise led by Insight Partners. CrewAI’s backers also include prominent AI researcher Andrew Ng and HubSpot Inc. co-founder Dharmesh Shah.

Many enterprise machine learning projects revolve around so-called AI agents. Those are large language models customized to perform a narrow set of tasks. An e-commerce company, for example, could cre

In [59]:
put(13,status='private',total_funding_usd='38000000',funding_asof='2026-04',funding_source=cite_result('crew_primary2',3)+'|'+cite_result('crew2',0),last_valuation_usd='185000000',valuation_date='2026-04',valuation_source=cite_result('crew_valuation',0),peak_value_usd='185000000',peak_value_date='2026-04',peak_source=cite_result('crew_valuation',0),ever_1b='no',notes='Investor discloses $185m post-money Series B; $18m inception/Series A plus $20m Series B counted; database totals conflict.')
print('Updated CrewAI',rows[13])
print('First Crew search primary results:')
for i,x in enumerate(searches['crew_primary']['results']):print(i,x['url'],(x.get('full_text') or '')[:700])

Updated CrewAI {'github_orgs': 'crewAIInc', 'entity': 'CrewAI', 'status': 'private', 'status_date': '', 'acquirer': '', 'exit_value_usd': '', 'exit_source': '', 'total_funding_usd': '38000000', 'funding_asof': '2026-04', 'funding_source': 'https://boldstart.vc/companies/crewai/|https://pitchbook.com/profiles/company/590845-78', 'last_valuation_usd': '185000000', 'valuation_date': '2026-04', 'valuation_source': 'https://earlgrey.capital/updates', 'peak_value_usd': '185000000', 'peak_value_date': '2026-04', 'peak_source': 'https://earlgrey.capital/updates', 'ever_1b': 'no', 'notes': 'Investor discloses $185m post-money Series B; $18m inception/Series A plus $20m Series B counted; database totals conflict.'}
First Crew search primary results:
0 https://www.insightpartners.com/ideas/crewai-launches-multi-agentic-platform-to-deliver-on-the-promise-of-generative-ai-for-enterprise/ CrewAI Launches Multi-Agentic Platform to Deliver on the Promise of Generative AI for Enterprise | Insight Partn

In [60]:
for k,task in tasks.items():
 s=await check_task(task);print(k,s.status,s.runtime_minutes,s.steps)
 if s.answer:Path(f'audit_work/{k}_agent_answer.md').write_text(s.answer);print(s.answer[:1000])

Task ddd0f559-7ce3-4b14-bb21-d0beeb9bf66a still running (status: in progress). Call check_task(task_id) again later.
smallA in progress 4.875310466666666 23


Task 08d1321c-1746-402b-bad9-7f630a43bf59 still running (status: in progress). Call check_task(task_id) again later.
smallB in progress 4.716748766666667 25


In [61]:
jobs={'anomaly_site':'https://anoma.ly','tiny_site':'https://tinygrad.org/','mermaid_site':'https://mermaid.ai/','tiger_site':'https://www.tigerdata.com/','portainer_site':'https://www.portainer.io/'}
r=await asyncio.gather(*(get(k,u) for k,u in jobs.items()),return_exceptions=True)
for k,t in zip(jobs,r):
 print('\n###',k)
 if isinstance(t,Exception):print(type(t).__name__);continue
 print(t[:2200])


### anomaly_site
Anoma.ly 

[![Anomaly logo variations](https://anoma.ly/anomaly-logo-flip.gif)](https://x.com/anomalyco) 

[Creators of ![OpenCode](https://anoma.ly/opencode-wordmark-dark.svg)](https://opencode.ai) 

# Looking for anomalies.

Experiments, experiences, and ambitious ideas from Anomaly.

[01 / Entertainment ## Rebase Live trivia. Real prizes. Play Rebase](https://rebase.tv)   [02 / Adventure ## Antarctica A bike challenge at the edge of the world. Follow the journey](https://www.instagram.com/reel/Daf-lKUz7iD/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA==)   [03 / Partnerships ## Let's connect We partner with incredible people doing incredible things. Talk to our team](mailto:help@anoma.ly)



### tiny_site
the new tinybox — tiny corp 

[docs](https://docs.tinygrad.org/tinybox/) [FAQ](https://tinygrad.org/faq.html)

# the new tinybox.

A computer for deep learning. Configured by you.

![Tall front view](https://tinygrad.org/assets/tinybox-tower/front.webp "width=63

In [62]:
def cite_page(label,author,date=None):
 x=pages[label];url=x['url']
 if url in used:return url
 title=x['text'].splitlines()[0].strip()
 title=title.replace('&',r'\&').replace('%',r'\%').replace('_',r'\_').replace('#',r'\#')
 key='audit'+str(len(used)+1)
 bib=f'@online{{{key},\n author = {{{{{author}}}}},\n title = {{{title}}},\n url = {{{url}}},\n urldate = {{2026-11-26}},\n'
 if date:bib+=f' date = {{{date}}},\n'
 bib+='}\n';save_citation(bib)
 used[url]={'title':title,'url':url,'full_text':x['text'],'author':author,'publishedDate':date,'key':key}
 Path('audit_work/used_sources.json').write_text(json.dumps(used,indent=2,default=str));return url
for label,author in [('anomaly_site','Anomaly'),('tiny_site','Tiny Corp'),('mermaid_site','Mermaid Chart'),('tiger_site','Tiger Data'),('portainer_site','Portainer')]:cite_page(label,author)
# Switch CrewAI total's historical component to explicit company press release mirrored by the investor.
rows[13]['funding_source']=cite_result('crew_primary',0)+'|'+cite_result('crew2',0)
pd.DataFrame([rows[i] for i in sorted(rows)],columns=cols).to_csv('audit_work/my_rows.csv',index=False)
print('Saved citations for',len(used),'used sources')

Saved citations for 46 used sources


In [63]:
t=await get('coder_forge',searches['coder']['results'][4]['url'])
for m in re.finditer(r'(?i)(?:post.money valuation|3\.|information|source|estimate|caplight|pitchbook)',t):
 p=m.start()
 if 'valuation' in t[max(0,p-20):p+40].lower() or p>len(t)-17000:
  print(t[max(0,p-100):p+700])
print('Tail',t[-5500:])

et Activity 2

Limited

Last Matched Price

[Get Full Access](https://auth.forgeglobal.com/signup)

Post-Money Valuation 3

$382.58M

Total Funding

$175.11M

Last Funding Round

Series C

## Learn more about Coder

### To invest in Coder pre-IPO

### Can you invest in Coder pre-IPO?

Coder is privately held and not traded on public markets, so pre-IPO investments are available only to accredited investors.

**If you’re already accredited and have completed Forge’s verification**, you can visit our [marketplace](https://auth.forgeglobal.com/) to review and invest in private companies ahead of an IPO.

**If you haven’t yet completed your accredited investor verification with Forge,** you can start [that process](https://auth.forgeglobal.com/) now to unlock access to new private-company offe
ls details

|                      | Funding Date | Round       | Amount Raised | Price per Share | Post-Money Valuation 3 | Key Investors        |
| -------------------- | ------------ | -----------

In [64]:
t=pages['coder_forge']['text']
for pat in ['3 Post','3.','Valuation data','valuation is','Post-Money Valuation','fully diluted','pre-money','derived','Footnote']:
 print('\nPAT',pat)
 for m in list(re.finditer(re.escape(pat),t,re.I))[-5:]:print(t[max(0,m.start()-120):m.start()+1000])


PAT 3 Post

PAT 3.
4 \| height=24")](https://forgeglobal.com/deel_stock/) | [Deel](https://forgeglobal.com/deel_stock/) | $32.49 \-$5.14 (13.66%) | $ xx.xx      | Series E           | $17.3B               | $38.92          | $300M         |
|               | [![Cohesity](https://forgeglobal.com/site/assets/files/1484/cohesity-logo-500w-1.48x48.png "width=24 \| height=24")](https://forgeglobal.com/cohesity_stock/) | [Cohesity](https://forgeglobal.com/cohesity_stock/) | $15.67               | $ xx.xx      | Series H-1         | $4.69B               | $15.66          | $27.5M        |
|               | [![Genesys](https://forgeglobal.com/site/assets/files/5719/genesys-logo-250w.48x48.png "width=24 \| height=24")](https://forgeglobal.com/genesys_stock/) | [Genesys](https://forgeglobal.com/genesys_stock/) | Not available        | $ xx.xx      | Not available      | \--                  | \--             | \--           |
|               | [![Relativity](https://forgeglobal.com/site/assets/

In [65]:
t=await get('deepseek_yicai',searches['deepseek']['results'][3]['url'])
p=t.find('# China'); print(t[p:p+12000])

)


In [66]:
print(pages['deepseek_yicai']['text'][:14500])
cite_page('coder_forge','Forge Global')
for k,task in tasks.items():
 s=await check_task(task);print(k,s.status,s.runtime_minutes,s.steps)
 if s.answer:Path(f'audit_work/{k}_agent_answer.md').write_text(s.answer);print(s.answer[:1500])

China’s DeepSeek Swells Investor Base in Second Fundraiser

![](https://www.yicaiglobal.com/6cc651869f31e4a06dfee7cc97566f81.svg)

[![](https://www.yicaiglobal.com/img/logo.478eb1fe.png)](https://www.yicaiglobal.com/)

![](https://www.yicaiglobal.com/img/head.30a9e378.png)

![](https://www.yicaiglobal.com/img/head.30a9e378.png)

[account inf](https://www.yicaiglobal.com/personal)

log out

LOG IN| [ABOUT US](https://www.yicaiglobal.com/about)|[CONTACT](https://www.yicaiglobal.com/contacts)

* [home](https://www.yicaiglobal.com/)
* [Economy](https://www.yicaiglobal.com/economy)
* [Business](https://www.yicaiglobal.com/business)
* [Tech](https://www.yicaiglobal.com/tech)
* [auto](https://www.yicaiglobal.com/auto)
* [Opinion](https://www.yicaiglobal.com/opinion)

[![](https://www.yicaiglobal.com/aefd51d689787ca026bc3849cb0b7606.svg)](https://www.facebook.com/yicaiglobal) [![](https://www.yicaiglobal.com/80966ca6cfb64e3e856765be6586d3a0.svg)](https://twitter.com/yicaichina) [![](https://ww

Task ddd0f559-7ce3-4b14-bb21-d0beeb9bf66a still running (status: in progress). Call check_task(task_id) again later.
smallA in progress 5.476977916666667 25


smallB success 6.443047966666667 26
## Files Ready for Analysis

| File | Description | Format | Size |
|---|---|---|---:|
| `financial_disclosures.csv` | Analysis-ready data: 11 companies, exact requested columns | CSV | 2,801 bytes |
| `evidence.json` | Supporting search results, fetched pages, queries, URLs and source metadata | JSON | 871,909 bytes |
| `sources.bib` | Supporting bibliography: 29 sources | BibTeX | 7,233 bytes |
| `methodology.json` | Supporting definitions, uncertainties and validation results | JSON | 2,796 bytes |
| `reproduce_retrieval.py` | Supporting script to repeat the recorded searches and fetches | Python | 999 bytes |

**Completed: 11 rows. Status counts: 10 private; 1 shut_down; 0 acquired, listed or unknown.** Research cutoff: **2026-11-26**. Each company received three to five searches or fetches. Nothing was uploaded.

Blank financial fields mean undisclosed or unverified, not zero. `ever_1b=no` follows your small-company screening rule; it does not e

In [67]:
ansB=Path('audit_work/smallB_agent_answer.md').read_text();print(ansB)

## Files Ready for Analysis

| File | Description | Format | Size |
|---|---|---|---:|
| `financial_disclosures.csv` | Analysis-ready data: 11 companies, exact requested columns | CSV | 2,801 bytes |
| `evidence.json` | Supporting search results, fetched pages, queries, URLs and source metadata | JSON | 871,909 bytes |
| `sources.bib` | Supporting bibliography: 29 sources | BibTeX | 7,233 bytes |
| `methodology.json` | Supporting definitions, uncertainties and validation results | JSON | 2,796 bytes |
| `reproduce_retrieval.py` | Supporting script to repeat the recorded searches and fetches | Python | 999 bytes |

**Completed: 11 rows. Status counts: 10 private; 1 shut_down; 0 acquired, listed or unknown.** Research cutoff: **2026-11-26**. Each company received three to five searches or fetches. Nothing was uploaded.

Blank financial fields mean undisclosed or unverified, not zero. `ever_1b=no` follows your small-company screening rule; it does not establish a historical valuation ceil

In [68]:
rows[26].update(last_valuation_usd='92000000000',valuation_date='2026-11',valuation_source=cite_result('deepseek',3),peak_value_usd='92000000000',peak_value_date='2026-11',peak_source=cite_result('deepseek',3),notes='Reported approximate USD funding; Yicai reports $92b post-money, versus Bloomberg $86b implied mark. No official financing terms released.')
# Parse the returned CSV directly from the agent answer (the remote agent filesystem is not mounted here).
import io
b_csv=re.search(r'```csv\n(.*?)\n```',ansB,re.S).group(1)
bdf=pd.read_csv(io.StringIO(b_csv),dtype=str).fillna('')
print(bdf[['github_orgs','status','total_funding_usd','ever_1b']].to_string(index=False))
for _,d in bdf.iterrows():
 i=src.index[src.github_orgs.eq(d.github_orgs)][0]
 rows[int(i)]=d.to_dict()
print('Integrated',len(bdf),'rows. Total',len(rows))
# Persist the agent's source citations with the required citation API in this workspace as well.
b_bibs=re.findall(r'```(?:bibtex|bib)\n(.*?)\n```',ansB,re.S)
print('Bib blocks',len(b_bibs))
for bib in b_bibs:save_citation(bib)
pd.DataFrame([rows[i] for i in sorted(rows)],columns=cols).to_csv('audit_work/combined_progress.csv',index=False)

github_orgs    status total_funding_usd ever_1b
 Textualize shut_down                        no
  OtterMind   private                   unknown
 heroui-inc   private                        no
   chatwoot   private           1600000      no
      typst   private                        no
   nanocoai   private          12000000      no
   usablica   private                        no
   LiteFold   private                        no
     aipoch   private                        no
    meshpro   private                        no
 borneo-iot   private                        no
Integrated 11 rows. Total 30
Bib blocks 1


In [69]:
print(ansB[ansB.find('### 2. OtterMind'):ansB.find('## Full BibTeX') if '## Full BibTeX' in ansB else ansB.find('```bibtex')])

### 2. OtterMind

**Status: private, operating; corporate attribution requires caution.** The [GitHub organization](https://github.com/ottermind) connects OtterMind with Chat2DB and the OtterMind website. The company's [About page](https://ottermind.ai/about) states: “Ottermind is operated by the Singapore-registered company listed here,” identifying **OTTERMIND PTE. LTD.** Its current site markets an operating AI-agent workspace.

**Financing conflict:** The [36Kr Chat2DB profile](https://pitchhub.36kr.com/project/3173327576230918) instead identifies the Chinese legal entity **爱獭科技（杭州）有限公司**, translated there as Love Otter Technology (Hangzhou) Co., Ltd. Its financing table lists angel, Series A and Series A+ events, with the latest in May 2026, but the amount field is **“未透露”**, meaning “undisclosed.” I did not establish whether these financings belong to the Singapore entity, an affiliate, or another corporate perimeter. No amount or valuation was transferred into OtterMind's row.



In [70]:
jobs={'textualize_shutdown':'https://textual.textualize.io/blog/2025/05/07/the-future-of-textualize/','chatwoot_funding':rows[23]['funding_source'],'nanoco_funding':rows[28]['funding_source'],'crew_primary':searches['crew_primary']['results'][0]['url']}
r=await asyncio.gather(*(get(k,u) for k,u in jobs.items()),return_exceptions=True)
for k,t in zip(jobs,r):
 print('\n###',k)
 if isinstance(t,Exception):print(type(t).__name__,str(t)[:200]);continue
 print('\n'.join(m.group(0) for m in re.finditer(r'.{0,110}(?:\$|wrapping|shut|total funding|raised).{0,210}',t))[:5500])


### textualize_shutdown
shared problem or pain-point to build a viable business around. Which is why Textualize, the company, will be wrapping up in the next few weeks.

### chatwoot_funding
### $1.6M Seed round
We raised a $1.6M seed round from [Goat Capital](https://goat.vc), [Uncorrelated Ventures](https://uncorrelated.com/) & [YCombinator](http://ycombinator.com).

### nanoco_funding
 we're announcing NanoCo (creators of NanoClaw), our enterprise offering from the team behind NanoClaw, and a $12M seed round led by Valley Capital Partners, with participation from Docker, Inc,… | NanoClaw       
edin.com/company/nanocoai?trk=public_post-text), our enterprise offering from the team behind NanoClaw, and a $12M seed round led by [Valley Capital Partners](https://www.linkedin.com/company/valley-capital-partners?trk=public_post-text), with participation from [Docker, Inc](https://www.linkedin.com/company/docker?trk=

### crew_primary
RqxvGvM08RhIWGv6WcpOt6nbc_JyayIYV7-1_vaJJxQT_P-Q== "Cr

In [71]:
s=await check_task(tasks['smallA']);print(s.status,s.runtime_minutes,s.steps)
if s.answer:
 Path('audit_work/smallA_agent_answer.md').write_text(s.answer); print(s.answer[:4500])
print('UTC',datetime.datetime.now(datetime.timezone.utc).isoformat())

success 6.923860583333334 26
## Files Ready for Analysis

| File | Description | Format | Size |
|---|---|---|---:|
| `financial_disclosures.csv` | Analysis-ready rows in the exact requested schema | CSV | 2,499 bytes |
| `evidence.json` | Supporting source extracts, search results, currency data, and caveats | JSON | 1,089,057 bytes |
| `sources.bib` | Complete supporting bibliography | BibTeX | 5,325 bytes |
| `provenance.json` | Supporting source and file manifest | JSON | 9,700 bytes |
| `retrieve.py` | Reproduction script with the reviewed snapshot embedded | Python | 13,283 bytes |

**Completed: 10 rows. Status counts: 9 private, 1 acquired, 0 listed, 0 shut down, 0 unknown.** Research cutoff: **2026-11-26**. The complete CSV and bibliography are reproduced below; no filesystem access is required to use this answer.

### Saved-file provenance

- `/workspace/downloads/oss_financial_disclosures/financial_disclosures.csv`: The ten reviewed company rows. Sources are the company-speci

In [72]:
ansA=Path('audit_work/smallA_agent_answer.md').read_text();print(ansA[:14500])

## Files Ready for Analysis

| File | Description | Format | Size |
|---|---|---|---:|
| `financial_disclosures.csv` | Analysis-ready rows in the exact requested schema | CSV | 2,499 bytes |
| `evidence.json` | Supporting source extracts, search results, currency data, and caveats | JSON | 1,089,057 bytes |
| `sources.bib` | Complete supporting bibliography | BibTeX | 5,325 bytes |
| `provenance.json` | Supporting source and file manifest | JSON | 9,700 bytes |
| `retrieve.py` | Reproduction script with the reviewed snapshot embedded | Python | 13,283 bytes |

**Completed: 10 rows. Status counts: 9 private, 1 acquired, 0 listed, 0 shut down, 0 unknown.** Research cutoff: **2026-11-26**. The complete CSV and bibliography are reproduced below; no filesystem access is required to use this answer.

### Saved-file provenance

- `/workspace/downloads/oss_financial_disclosures/financial_disclosures.csv`: The ten reviewed company rows. Sources are the company-specific URLs below. Transformatio

In [73]:
print(ansA[ansA.find('### 1. datalab'):ansA.find('### 5. VectifyAI')])
a_csv=re.search(r'```csv\n(.*?)\n```',ansA,re.S).group(1)
adf=pd.read_csv(io.StringIO(a_csv),dtype=str).fillna('')
for _,d in adf.iterrows():
 i=src.index[src.github_orgs.eq(d.github_orgs)][0];rows[int(i)]=d.to_dict()
for bib in re.findall(r'```(?:bibtex|bib)\n(.*?)\n```',ansA,re.S):save_citation(bib)
print('All rows:',len(rows))
# Do not infer an exact closing month from an approximate relative-date statement.
rows[8]['funding_asof']=''
rows[8]['notes']='Founder disclosed $3m in August 2026, raised about a year earlier; exact round month undisclosed. Valuation undisclosed.'
fxurl='https://api.frankfurter.dev/v2/rates?date=2025-03-21&base=GBP&quotes=USD&providers=ecb&expand=providers'
rows[7]['funding_source']+='|'+fxurl
pd.DataFrame([rows[i] for i in range(40)],columns=cols).to_csv('finance_audit.csv',index=False)
print('Wrote full finance_audit.csv')

### 1. datalab-org — datalab industries ltd

**Status: private and operating.** The [company website](https://datalab.industries/) describes itself as “The open source scientific software and data consultancy” and offers “Managed hosting of datalab instances.” [Companies House](https://find-and-update.company-information.service.gov.uk/company/15704476) identifies DATALAB INDUSTRIES LTD as “Active” and a “Private limited Company.”

No qualifying equity-funding or valuation disclosure was located. Academic project support and consultancy income were not treated as equity financing.

### 2. ankitects — Anki Software, LLC (Ankitects)

**Status: acquired by AnkiHub; the business remains operating.** The [February 2, 2026 announcement](https://forums.ankiweb.net/t/ankis-growing-up/68610) states:

> “I ended up suggesting to them that we look into gradually transitioning business operations and open source stewardship over”

The same discussion states:

> “Both Anki and AnkiHub are entirely 

In [74]:
jobs={'anki_transition':'https://forums.ankiweb.net/t/ankis-growing-up/68610','anki_paid':'https://forums.ankiweb.net/t/regarding-recent-events/74723','bruno_round':rows[8]['funding_source'],'vectify_round':rows[7]['funding_source'].split('|')[0],'fx_conversion':fxurl}
r=await asyncio.gather(*(get(k,u) for k,u in jobs.items()),return_exceptions=True)
for k,t in zip(jobs,r):
 print('\n###',k)
 if isinstance(t,Exception):print(type(t).__name__,str(t)[:180]);continue
 if k=='fx_conversion':print(t[:2000]);continue
 print('\n'.join(m.group(0) for m in re.finditer(r'.{0,120}(?:\$|million|instal|transition|pay me|202[5-6]|self-funded|year ago).{0,200}',t))[:9000])


### anki_transition
s better positioned to take Anki to the next level than I am. I ended up suggesting to them that we look into gradually transitioning business operations and open source stewardship over, with provisions in place to ensure that Anki remains open source and true to the principles I’ve run it by all these years.

### anki_paid
Full disclosure up front: the deal I made with AnkiHub requires them to pay me over time. They’ll only be able to do that if Anki remains profitable. That makes me hesitate to post this, because it’s in my best interests that Anki do well.
When I reflect on the way the management transition was handled on my end, I’m ashamed that I did not handle it better. Ideally it would have been done over a longer time frame, giving the community time to get to know the AnkiHub team, and for things
 to transition over smoothly. Sadly, that’s not how it worked out - I rushed the transition through because I was really struggling to hold on any longer.
My su

In [75]:
print(ansA[ansA.find('@misc{vectify'):ansA.find('@misc{marmelab')])
print('Notes >25:',[(i,len(d['notes'].split()),d['notes']) for i,d in rows.items() if len(d['notes'].split())>25])
print('Status counts:',pd.Series([d['status'] for d in rows.values()]).value_counts().to_dict())
print('ever counts:',pd.Series([d['ever_1b'] for d in rows.values()]).value_counts().to_dict())
print('All numbers:')
print(pd.DataFrame([rows[i] for i in range(40)])[['entity','total_funding_usd','funding_asof','last_valuation_usd','peak_value_usd','ever_1b']].to_string(index=False))

@misc{vectify_about,
  author = {{Vectify AI}},
  title = {About PageIndex},
  url = {https://pageindex.ai/about},
  urldate = {2026-11-26}
}

@misc{vectify_funding,
  author = {{Bae HQ}},
  title = {Vectify AI Raises £1.1M to Automate Document and Financial Data Analysis},
  url = {https://www.thebaehq.com/news/vectify-ai-raises-1-1m-to-automate-document-and-financial-data-analysis},
  urldate = {2026-11-26},
  date = {2025-03-21}
}

@misc{vectify_conflict,
  author = {{Dealroom}},
  title = {PageIndex company information, funding and investors},
  url = {https://app.dealroom.co/companies/pageindex},
  urldate = {2026-11-26}
}

@misc{gbp_usd,
  author = {{Frankfurter and European Central Bank}},
  title = {GBP/USD reference exchange rate, 21 March 2025, ECB provider},
  url = {https://api.frankfurter.dev/v2/rates?date=2025-03-21&base=GBP&quotes=USD&providers=ecb&expand=providers},
  urldate = {2026-11-26},
  date = {2025-03-21}
}

@misc{fx_documentation,
  author = {{Frankfurter}},
  

In [76]:
rows[4]['notes']='Paid operating-business transfer to AnkiHub; price undisclosed. February 2026 is announcement, not verified closing. Legal-entity transaction scope unverified.'
# Citation records for compared/rejected public figures, kept distinct from accepted numeric fields.
conflict_sources={
 'anomaly_db':cite_result('anomaly',1),
 'penpot_db':cite_result('penpot',2),
 'crew_db':cite_result('crew_valuation',3),
 'crew_forge':cite_result('crew_valuation',1),
 'continue_db':cite_result('continue2',0),
 'continue_linkedin':cite_result('continue',2),
 'coder_alt':cite_result('coder2',1),
 'portainer_db':cite_result('portainer',4),
 'portainer_old_db':cite_result('portainer',2),
 'strapi_db':cite_result('strapi',0),
 'tile_alt':cite_result('tiledb',1),
 'tile_db':cite_result('tiledb',0),
 'mermaid_db':cite_result('mermaid',0),
 'mermaid_dealroom':cite_result('mermaid',4),
 'tiger_db':cite_result('tiger',2),
 'rocket_db':cite_result('rocket',1),
 'rocket_alt':cite_result('rocket',3),
 'deepseek_bloomberg':cite_result('deepseek',4),
}
print('All notes valid',all(len(d['notes'].split())<=25 for d in rows.values()))
print('Now',len(used),'own used source citations, plus 53 agent citations.')

All notes valid True
Now 64 own used source citations, plus 53 agent citations.


In [77]:
conflicts=[]
def conflict(idx,topic,a,au,b,bu,resolution,kind='conflicting_report'):
 conflicts.append({'github_orgs':src.at[idx,'github_orgs'],'entity':src.at[idx,'entity'],'topic':topic,'source_a_statement':a,'source_a_url':au,'source_b_statement':b,'source_b_url':bu,'resolution':resolution,'discrepancy_type':kind})
conflict(0,'Funding total','Company announced $1m seed, including YC',rows[0]['funding_source'],'aVenture reports $1.1m cumulative',conflict_sources['anomaly_db'],'Count $1m only; no separately verified additional amount.','coverage_or_rounding')
conflict(6,'Round label','Company calls $1.1m round seed',rows[6]['funding_source'],'Tech.eu calls same round pre-seed','https://tech.eu/2023/09/29/pandasai-raises-11m-in-pre-seed-round-for-ai-powered-data-analysis','Amount unaffected; label disagreement retained in notes.','round_label_only')
conflict(7,'Round amount and history','Bae HQ reports GBP1.1m in March 2025',rows[7]['funding_source'].split('|')[0],'Dealroom lists GBP1.2m in March and GBP0.3m in July 2025','https://app.dealroom.co/companies/pageindex','Use reported GBP1.1m converted at contemporaneous ECB rate; additional round uncorroborated. Bae HQ itself cites Crunchbase.','amount_and_coverage')
conflict(12,'Cumulative funding','Nordic9 reports USD30.54m total and USD7.93m July 2026 round',rows[12]['funding_source'],'CB Insights reports USD22.59m cumulative including July 2026 USD7.89m round',conflict_sources['penpot_db'],'Prefer Nordic9; company timeline confirms distinct USD8m 2022 and USD12m 2023 rounds, while SETT confirms EUR6.9m 2026 round.','amount_and_coverage')
conflict(13,'Cumulative funding','Issuer/investor says USD18m TOTAL through inception and Series A; PitchBook lists USD20m Series B',rows[13]['funding_source'],'PitchBook reports USD44.5m raised to date',conflict_sources['crew_db'],'Count USD18m + USD20m = USD38m; do not add seed again to an explicitly cumulative USD18m figure. Series B confirmed by participating investors.')
conflict(13,'Private valuation','Earl Grey Capital states USD185m POST-MONEY Series B',rows[13]['valuation_source'],'Forge shows USD177.15m valuation',conflict_sources['crew_forge'],'Prefer participating investor disclosure; Forge labels post-money figures as estimates derived from incorporation certificates.','disclosed_mark_vs_estimate')
conflict(14,'Cumulative funding','Company release says total USD5m seed funding',rows[14]['funding_source'],'Tracxn USD5.6m; LinkedIn financial panel USD7.325m',conflict_sources['continue_db']+'|'+conflict_sources['continue_linkedin'],'Prefer company-issued cumulative USD5m figure; do not double count total as another round.')
conflict(19,'Cumulative funding and valuation','CEO interview USD85m through 2024 plus company USD90m 2026 round',rows[19]['funding_source'],'Contrary reports USD172.8m cumulative and cites unverified USD828m 2026 valuation; Forge USD175.11m funding, USD382.58m estimated post-money',conflict_sources['coder_alt']+'|'+pages['coder_forge']['url'],'Use rounded USD175m funding sum. Leave valuation blank: current issuer announcement gives none and alternative marks are explicitly estimated/unverified.','amount_and_estimates')
conflict(20,'Cessation date','Founder announced operational wind-down in May 2025','https://textual.textualize.io/blog/2025/05/07/the-future-of-textualize/','PitchBook dates Out of Business event February 2026','https://pitchbook.com/profiles/company/510364-36','Use May 2025 announcement month, not a claimed legal dissolution/completion date.','announcement_vs_database_event')
conflict(22,'Cumulative funding','LinkedIn financial panel USD425000','https://linkedin.com/company/heroui','aVenture USD500000','https://aventure.vc/companies/heroui-san-francisco-ca-us/fundraising','Leave amount blank. Investor Epakon verifies pre-seed participation but no amount; generic YC terms not substituted.')
conflict(23,'Cumulative funding','Company says USD1.6m seed including YC',rows[23]['funding_source'],'PitchBook says USD1.91m','https://pitchbook.com/profiles/company/227991-43','Prefer explicit company figure, corroborated by VentureBeat; no generic accelerator cheque added.')
conflict(25,'Cumulative funding history','Disclosed USD20m through 2020 plus USD34m Series B in 2023',rows[25]['funding_source'],'Airframe USD53.1m, CB Insights USD54.07m',conflict_sources['tile_alt']+'|'+conflict_sources['tile_db'],'Count USD54m through October 2023. Company-confirmed September 2026 strategic financing has no disclosed amount.','amount_and_coverage')
conflict(26,'Post-money valuation and round size','Yicai: reported CNY105.5b round; CNY615.5b / about USD92b post-money',rows[26]['valuation_source'],'Bloomberg/Straits Times: about CNY100b raised and USD86b implied valuation',conflict_sources['deepseek_bloomberg'],'Use reported round post-money USD92b, also highest explicit round mark found. Do not replace it with a later registry-implied estimate. Funding sum uses Reuters USD7.4b first round plus Caixin about USD15b second round.','reported_terms_vs_implied_mark')
conflict(27,'Cumulative funding and later rounds','Issuer announcements disclose USD1.2m seed + USD6m Series A + USD6.2m extension',rows[27]['funding_source'],'PitchBook USD18.3m total and USD4.55m September 2025 round; Tracxn only USD7.2m',conflict_sources['portainer_db']+'|'+conflict_sources['portainer_old_db'],'Count USD13.4m through June 2022; later database-only round and different earlier amount uncorroborated.','amount_and_coverage')
conflict(29,'Equity versus mixed funding','Company equity history USD31m + USD10m + USD4m + USD1m = USD46m',rows[29]['funding_source'],'CB Insights USD49.76m cumulative, listing a loan and later USD1.27m VC entry',conflict_sources['strapi_db'],'Count company equity history only; exclude debt and unverified later entries or secondary sales.','funding_scope')
conflict(32,'Equity versus convertible financing','Company announces USD7.5m seed',rows[32]['funding_source'],'CB Insights USD9.5m including earlier convertible note; Dealroom labels earlier USD2m as seed',conflict_sources['mermaid_db']+'|'+conflict_sources['mermaid_dealroom'],'Count disclosed USD7.5m equity round only; conversion of earlier note and amount of later 2025 financing not verified.','instrument_and_coverage')
conflict(38,'Cumulative funding precision','Company release USD180m cumulative; valuation OVER USD1b',rows[38]['funding_source'],'Clarity USD181.1m cumulative and rounded USD1b post-money',conflict_sources['tiger_db'],'Prefer company funding total. Exact valuation blank because issuer supplies a lower bound, not an exact mark; ever_1b=yes.','rounding_or_scope')
conflict(39,'Cumulative funding and valuation','TechCrunch USD27m cumulative 2021 plus investor counsel USD10m equity bridge 2023',rows[39]['funding_source'],'CB Insights USD48.02m including note; Caplight USD36.9m and USD30.9m ESTIMATED valuation',conflict_sources['rocket_db']+'|'+conflict_sources['rocket_alt'],'Count disclosed USD37m equity sum; leave valuation blank rather than substitute estimates or ambiguous database marks.','funding_scope_and_estimates')
cdf=pd.DataFrame(conflicts)
cdf.to_csv('finance_audit_conflicts.csv',index=False)
print('Saved',len(cdf),'discrepancies for',cdf.github_orgs.nunique(),'companies')

Saved 18 discrepancies for 17 companies


In [78]:
# Restore exact input labels and enforce the final input order.
for i in range(40):
 rows[i]['github_orgs']=src.at[i,'github_orgs'];rows[i]['entity']=src.at[i,'entity']
rows[7]['notes']='Reported £1.1m; converted at ECB 2025-03-21 GBP/USD 1.2925. Lower-confidence news cites Crunchbase; additional database rounds unverified.'
final_df=pd.DataFrame([rows[i] for i in range(40)],columns=cols)
final_df.to_csv('finance_audit.csv',index=False)

status_urls={
0:[pages['anomaly_site']['url']],1:[rows[1]['exit_source']],2:[searches['openbb3']['results'][0]['url']],
3:['https://datalab.industries/','https://find-and-update.company-information.service.gov.uk/company/15704476'],
4:['https://forums.ankiweb.net/t/ankis-growing-up/68610','https://forums.ankiweb.net/t/regarding-recent-events/74723'],
5:['https://notable.app/'],6:['https://www.ycombinator.com/companies/pandasai'],7:['https://pageindex.ai/about'],8:[rows[8]['funding_source']],
9:['https://marmelab.com/blog','https://marmelab.com/blog/2023/11/08/open-source-profit-1.html'],
10:['https://bruce.computer/','https://bruce.computer/boards'],11:[pages['tiny_site']['url']],12:[searches['penpot']['results'][0]['url']],
13:[searches['crew_round']['results'][1]['url'],searches['crew_primary']['results'][0]['url']],14:[rows[14]['exit_source']],
15:[searches['unsloth2']['results'][4]['url'],searches['unsloth']['results'][4]['url']],16:[searches['qdrant']['results'][0]['url']],
17:['https://blog.gitea.com/gitea-cloud/','https://blog.gitea.com/a-message-from-lunny-on-gitea-ltd.-and-the-gitea-project/'],
18:['https://grapesjs.com/','https://opencoreventures.com/insights/grapesjs-creator-founds-grapes-studio-inc-to-bring-the-open-source-multi-project-no-code-website-editor-mainstream/'],
19:[searches['coder']['results'][1]['url']],20:['https://textual.textualize.io/blog/2025/05/07/the-future-of-textualize/'],
21:['https://ottermind.ai/about','https://github.com/ottermind'],22:['https://www.ycombinator.com/companies/heroui'],23:['https://www.chatwoot.com/'],
24:['https://typst.app/about/','https://typst.app/blog/2025/future/'],25:[searches['tiledb']['results'][2]['url']],26:[searches['deepseek']['results'][3]['url']],
27:[pages['portainer_site']['url']],28:['https://www.nanoco.ai/'],29:['https://strapi.io/blog/series-b'],30:['https://introjs.com/'],
31:[searches['avalonia']['results'][0]['url'],searches['avalonia']['results'][1]['url']],32:[pages['mermaid_site']['url']],33:[searches['openfaas2']['results'][2]['url']],
34:['https://www.litefold.ai/','https://yourstory.com/2026/05/litefold-biotech-startup-using-proprietary-ai-speed-up-drug-discovery'],
35:['https://www.prnewswire.com/news-releases/aipoch-launches-open-science-making-research-easier-to-follow-learn-from-and-share-with-science-packages-302908207.html'],
36:['https://mondaytech.com/meshpro/getting-started/','https://github.com/meshpro/optimesh'],
37:['https://www.binarystarstech.com/en/about/','https://www.borneoiot.com/app/tos/'],38:[pages['tiger_site']['url']],39:[searches['rocket']['results'][4]['url']]}

components={
0:{'usd_components':[1000000],'basis':'Company-announced seed only; YC participates in this round, so no generic accelerator cheque added.'},
1:{'usd_components':[8000000],'basis':'Company-announced seed before acquisition.'},2:{'usd_components':[8500000],'basis':'Company-issued seed announcement.'},
6:{'usd_components':[1100000],'basis':'Company-announced round only. Additional YC backing is confirmed without a company-specific amount.'},
7:{'original_currency':'GBP','original_amount':1100000,'usd_per_gbp':1.2925,'fx_date':'2025-03-21','fx_url':fxurl,'calculation':'1100000 * 1.2925 = 1421750','basis':'Bae HQ report relies on Crunchbase; treated as lower-confidence reported funding, not a company announcement.'},
8:{'usd_components':[3000000],'basis':'Founder announced August 11 2026, saying about a year ago. Exact round month not inferred, so funding_asof is blank.'},
11:{'usd_components':[5100000],'basis':'Founder headline states $5.1m; article body rounds to $5m. Proposed 2026 fundraising excluded.'},
12:{'reported_cumulative_usd':30540000,'latest_original_currency':'EUR','latest_original_amount':6900000,'latest_round_usd_as_reported_by_nordic9':7930000,'basis':'Adopt published USD-equivalent cumulative total, rather than performing an undocumented FX conversion. Company timeline confirms 2022 and 2023 financings are distinct.'},
13:{'usd_components':[18000000,20000000],'calculation':'18000000 + 20000000 = 38000000','basis':'Issuer/investor explicitly labels $18m cumulative through inception and Series A. PitchBook provides $20m April 2026 Series B; participating investors independently confirm Series B event and $185m post-money mark. Do not add seed again.'},
14:{'reported_cumulative_usd':5000000,'basis':'Company-issued press release explicitly says total $5m seed funding.'},
16:{'reported_cumulative_usd':87800000,'basis':'TechTarget March 12 2026 reports cumulative $87.8m; company confirms latest $50m round.'},
19:{'usd_components':[85000000,90000000],'calculation':'85000000 + 90000000 = 175000000','basis':'Rounded $85m total from interview with CEO in September 2024 plus $90m company-announced Series C in April 2026. This is a sum of reported rounded values, not exact audited proceeds.'},
23:{'reported_cumulative_usd':1600000,'basis':'Company investor handbook names $1.6m seed including Y Combinator. August 2021 announcement corroborated by VentureBeat.'},
25:{'usd_components':[20000000,34000000],'calculation':'20000000 + 34000000 = 54000000','basis':'Contemporaneous news reports cumulative $20m through Series A; company announces $34m Series B. Additional 2026 strategic investment amount undisclosed, excluded.'},
26:{'usd_components':[7400000000,15000000000],'calculation':'7400000000 + 15000000000 = 22400000000','basis':'Reuters reports first external round $7.4b; Caixin reports second completed round about $15b in November. Values are approximate published USD equivalents, not exact proceeds. Yicai reports explicit $92b round post-money; Bloomberg later gives an $86b registry-implied mark, excluded as a different metric.'},
27:{'usd_components':[1200000,6000000,6200000],'calculation':'1200000 + 6000000 + 6200000 = 13400000','basis':'Company-issued 2021 release states $1.2m seed and $6m Series A; company announces a separate $6.2m extension in June 2022. Later database-only financing not independently corroborated.'},
28:{'usd_components':[12000000],'basis':'Company-announced May 2026 seed. Declined $20m offer is not an exit or post-money financing valuation.'},
29:{'usd_components':[1000000,4000000,10000000,31000000],'calculation':'1000000 + 4000000 + 10000000 + 31000000 = 46000000','basis':'Sum of company handbook equity history. Excludes loans and unverified secondary transactions.'},
32:{'usd_components':[7500000],'basis':'Company-announced seed. Earlier $2m is described inconsistently as convertible note or seed by databases; no conversion verified, so excluded.'},
38:{'reported_cumulative_usd':180000000,'basis':'Company-issued release states $180m cumulative. Valuation is disclosed only as over $1b. Exact last/peak valuation fields remain blank rather than encoding the lower bound as an exact value.'},
39:{'usd_components':[27000000,10000000],'calculation':'27000000 + 10000000 = 37000000','basis':'TechCrunch reports $27m cumulative in 2021; participating investor counsel confirms additional $10m equity bridge in September 2023. Later convertible note excluded.'}}

# Company evidence from delegated searches is returned inline, not inferred from inaccessible remote files.
agent_sections={}
for ans,inds in [(ansA,groups['smallA']),(ansB,groups['smallB'])]:
 matches=list(re.finditer(r'^### (\d+)\. ',ans,re.M))
 # Only numbered company headings are used; bibliography and remote-file lists are excluded.
 for j,m in enumerate(matches[:len(inds)]):
  end=matches[j+1].start() if j+1<len(matches) else len(ans)
  section=ans[m.start():end]
  section=re.split(r'\n## (?:Uncertain|Full BibTeX|Saved files|Interpretation)',section)[0].strip()
  agent_sections[inds[j]]=section

methodology={
 'cutoff_date':'2026-11-26',
 'scope':'40 input rows; input order and entity labels preserved. Public disclosures, not a paid database census or audited financial statements.',
 'schema':'User-requested 18-column schema overrides owner/ticker/legal_name columns in the generic rules file.',
 'empty_numeric_fields':'Undisclosed, not securely attributable, or not independently verified; never replaced with zero.',
 'funding_scope':'Disclosed equity rounds through funding_asof. Explicitly disclosed components can be added; no generic accelerator cheque inferred. A counted total may be a disclosed minimum if other rounds have no verified amount.',
 'funding_dates':'Funding announcement/round month is used when established. Bruno exact month is blank because only an approximate relative date was supplied.',
 'source_hierarchy':'Prefer issuer, investor and SEC disclosures; then reputable reporting. Reject modeled enterprise-value ranges, estimated secondary-market marks, and uncorroborated database-only events. The CrewAI Series B amount uses PitchBook after primary investor corroboration of that specific round.',
 'non_usd':'Vectify GBP round converted with contemporaneous ECB reference rate via Frankfurter. Penpot and DeepSeek use source-published USD equivalents; original currencies and reported approximation retained in evidence/notes.',
 'status_dates':'Listing/acquisition/shutdown event or public announcement month; no invented event date for private operating companies. Anki and Textualize dates are announcement dates, not independently established legal closing/dissolution dates.',
 'latest_and_peak':'Only qualifying disclosed round post-money marks, first-day closing market capitalizations, or consummated exit prices. Rejected offers and intended fundraising not counted; Tiger Data exact valuation blank because issuer supplied only an over-$1b bound.',
 'ever_1b':'yes requires a disclosed qualifying value or explicit bound reaching $1b; no follows the rules for all disclosed values below $1b or an operating business clearly small in the retrieved evidence; unknown where size/ownership perimeter or valuation history is insufficient. no does not prove a historical ceiling.',
 'limitations':['Bounded public-source searches cannot prove absence of undisclosed or unindexed financings.','No disclosed acquisition prices were verified for the three acquired rows.','Some cumulative funding figures differ because of rounding, financing instruments, corporate perimeter, or incomplete round coverage. See discrepancy_type in the companion conflict table.','Earl Grey Capital CrewAI update was readable in search-index text; full-page fetch returned HTTP 401. The $185m post-money statement was retained with this access limitation.','No market-cap estimates, revenue multiples, inferred accelerator valuations, or generic deal-size multiples were used.']}

# Collect bibliographic entries saved through the citation API for the two remote research batches.
agent_bibtex='\n'.join(re.findall(r'```(?:bibtex|bib)\n(.*?)\n```',ansA+'\n'+ansB,re.S))
source_library={u:dict(v) for u,v in used.items()}
# Add fetched source text only when a source was actually fetched successfully.
for lab,p in pages.items():
 u=p['url']
 if u in source_library:source_library[u]['fetched_text']=p['text'];source_library[u]['fetch_access']=p.get('access','full parsed page')

company_evidence=[]
for i,d in enumerate(final_df.to_dict('records')):
 urls=[]
 for field in ['exit_source','funding_source','valuation_source','peak_source']:
  if d[field]:urls.extend(d[field].split('|'))
 record={'input_index':i+1,'github_orgs':d['github_orgs'],'entity':d['entity'],'accepted_row':d,'status_source_urls':status_urls[i],
         'accepted_financial_source_urls':list(dict.fromkeys(urls)),
         'funding_basis':components.get(i,{'basis':'No qualifying disclosed equity amount verified. Blank is not zero.'}),
         'discrepancies':[x for x in conflicts if x['github_orgs']==d['github_orgs']]}
 if i in agent_sections:record['research_source_excerpts_and_decisions']=agent_sections[i]
 company_evidence.append(record)

evidence={'methodology':methodology,'companies':company_evidence,'source_library':source_library,'delegated_source_bibtex':agent_bibtex,
          'search_queries_performed_in_main_audit':[{ 'label':k,'query':v['query']} for k,v in searches.items()],
          'main_audit_additional_verified_pages':{k:v for k,v in pages.items() if k in ['anki_transition','anki_paid','textualize_shutdown','chatwoot_funding','nanoco_funding','bruno_round','vectify_round','fx_conversion']}}
Path('finance_audit_evidence.json').write_text(json.dumps(evidence,indent=2,ensure_ascii=False,default=str))
print('Created evidence for',len(company_evidence),'companies; size',Path('finance_audit_evidence.json').stat().st_size,'bytes')

Created evidence for 40 companies; size 1045127 bytes


In [79]:
import csv, hashlib
check=pd.read_csv('finance_audit.csv',dtype=str,keep_default_na=False)
assert len(check)==len(src)==40
assert list(check.columns)==cols
assert check.github_orgs.tolist()==src.github_orgs.tolist()
assert check.entity.tolist()==src.entity.tolist()
assert set(check.status).issubset({'listed','acquired','private','shut_down','unknown'})
assert set(check.ever_1b).issubset({'yes','no','unknown'})
num_source=[('exit_value_usd','exit_source'),('total_funding_usd','funding_source'),('last_valuation_usd','valuation_source'),('peak_value_usd','peak_source')]
for num,source in num_source:
 assert check[num].map(lambda x:not x or bool(re.fullmatch(r'[0-9]+',x))).all(),num
 assert check.loc[check[num].ne(''),source].ne('').all(),source
for date in ['funding_asof','valuation_date','peak_value_date']:
 assert check[date].map(lambda x:not x or bool(re.fullmatch(r'\d{4}-\d{2}',x))).all(),date
assert check.status_date.map(lambda x:not x or bool(re.fullmatch(r'\d{4}(?:-\d{2})?',x))).all()
assert check.notes.str.split().str.len().le(25).all()
assert check.loc[check.status.ne('private'),'last_valuation_usd'].eq('').all()
assert check.loc[check.status.eq('acquired'),'acquirer'].ne('').all()
for num,source in num_source:
 for value in check.loc[check[source].ne(''),source]:
  assert all(u.startswith(('https://','http://')) for u in value.split('|'))
status_counts={k:int((check.status==k).sum()) for k in ['listed','acquired','private','shut_down','unknown']}
ever_counts={k:int((check.ever_1b==k).sum()) for k in ['yes','no','unknown']}
print('STATUS COUNTS',status_counts)
print('EVER_1B COUNTS',ever_counts)
print('Rows with disclosed funding',int(check.total_funding_usd.ne('').sum()))
print('Acquired:',check.loc[check.status.eq('acquired'),'entity'].tolist())
print('Shut down:',check.loc[check.status.eq('shut_down'),'entity'].tolist())
print('Ever1b unknown:',check.loc[check.ever_1b.eq('unknown'),'entity'].tolist())
print('Source discrepancies:',cdf.entity.drop_duplicates().tolist())
print('Max note words',check.notes.str.split().str.len().max())

# Add machine-readable validation/provenance and the two headline count tables.
evidence['summary_counts']={'rows':40,'status':status_counts,'ever_1b':ever_counts,'discrepancy_rows':len(cdf),'companies_with_source_discrepancies':int(cdf.github_orgs.nunique())}
evidence['validation']={'passed':True,'checks':['Exact requested columns and input order','40 rows','Input entity labels preserved','Allowed status and ever_1b values','Financial amounts use plain integer USD digits or blank','Every nonblank financial amount has one or more source URLs','Date formats checked','All notes at most 25 whitespace-delimited words','Acquired rows have acquirer; nonprivate rows have no current private valuation'], 'maximum_notes_words':int(check.notes.str.split().str.len().max())}
evidence['provenance']={'input_sha256':hashlib.sha256(Path('finance_audit_sample.csv').read_bytes()).hexdigest(),'rules_sha256':hashlib.sha256(Path('finance_rules.md').read_bytes()).hexdigest(),'finance_audit_csv_sha256':hashlib.sha256(Path('finance_audit.csv').read_bytes()).hexdigest(),'conflicts_csv_sha256':hashlib.sha256(Path('finance_audit_conflicts.csv').read_bytes()).hexdigest()}
Path('finance_audit_evidence.json').write_text(json.dumps(evidence,indent=2,ensure_ascii=False,default=str))
assert len(json.loads(Path('finance_audit_evidence.json').read_text())['companies'])==40

memory_path=Path('/workspace/.prompts/MEMORY.md')
mem=memory_path.read_text()
mem=mem.replace('## Context\n\n[Fill me in]','## Context\n\n- Financial-audit project: use the user-requested CSV schema when it differs from the generic rules file.')
mem=mem.replace('## Notes\n\n[Fill me in]','## Notes\n\n- Preserve input company order and labels; financial CSV amounts are plain USD integer digits, with undisclosed or unverified numbers blank, never zero-filled.\n- Accompany the audit with counts by status and ever_1b, and document conflicting sources, scope differences, and discretionary sourcing decisions.\n- In this deployment, data-retrieval subagents did not see local input files despite the shared-workspace description; pass small inputs inline and require complete CSV/evidence/BibTeX in the returned answer.')
memory_path.write_text(mem)

manifest={'files':[
 {'path':'/workspace/finance_audit.csv','description':'Financial audit of all 40 input companies in input order, with the exact requested 18 columns and source URLs.','kind':'final'},
 {'path':'/workspace/finance_audit_conflicts.csv','description':'18 documented source discrepancies across 17 companies, distinguishing contradictions from scope, timing and rounding differences.','kind':'final'},
 {'path':'/workspace/finance_audit_evidence.json','description':'Per-company status evidence, funding calculations, source extracts and metadata, methodological decisions, limitations, counts and validation results.','kind':'final'},
 {'path':'/workspace/audit_work/searches.json','description':'Executed main-audit search queries and original returned results.','kind':'intermediate'},
 {'path':'/workspace/audit_work/pages.json','description':'Successfully fetched source-page text and explicitly labeled access limitations.','kind':'intermediate'},
 {'path':'/workspace/audit_work/smallA_agent_answer.md','description':'Returned inline research evidence and bibliography for the first small-company batch.','kind':'intermediate'},
 {'path':'/workspace/audit_work/smallB_agent_answer.md','description':'Returned inline research evidence and bibliography for the second small-company batch.','kind':'intermediate'}]}
Path('.deliverables.json').write_text(json.dumps(manifest,indent=2))
for f in manifest['files']:
 assert Path(f['path']).exists(),f['path']
print('All final checks passed.')

STATUS COUNTS {'listed': 0, 'acquired': 3, 'private': 35, 'shut_down': 2, 'unknown': 0}
EVER_1B COUNTS {'yes': 2, 'no': 32, 'unknown': 6}
Rows with disclosed funding 21
Acquired: ['Kilo Code', 'Anki Software, LLC (Ankitects)', 'Continue Dev, Inc.']
Shut down: ['OpenBB Inc.', 'Textualize']
Ever1b unknown: ['Qdrant', 'Coder Technologies', 'OtterMind', 'TileDB, Inc.', 'Strapi', 'Rocket.Chat']
Source discrepancies: ['Anomaly', 'Sinaptik (PandasAI)', 'Vectify AI', 'Kaleidos (Penpot)', 'CrewAI', 'Continue Dev, Inc.', 'Coder Technologies', 'Textualize', 'HeroUI Inc.', 'Chatwoot', 'TileDB, Inc.', 'DeepSeek', 'Portainer.io', 'Strapi', 'Mermaid Chart', 'Tiger Data', 'Rocket.Chat']
Max note words 20
All final checks passed.
